# Qwen2-Audio Fine-Tuning for Moroccan Darija ASR
## Phase 1 Research Notebook (Marco-ASR adaptive LR) — v0.2

Implements the Phase 1 methodology for the PFE project: *Comparing End-to-End Audio-LLMs and Cascaded ASR for Darija Transcription*.

**v0.2 changes from v0.1:**
- Effective batch size 4 → **16** (`per_device=4`, `grad_accum=4`) — gives smoother gradients, better GPU utilization. Aligns with SpeechLLMs paper.
- Marco-ASR eval subset size 50 → **scale-tuned 200 / 500 / 1000** (10h / 30h / 80h). At 50, σ(WER) ≈ 2pp swamps the 0.5pp min-delta; the new sizes give σ ≤ min-delta.
- Marco-ASR eval frequency: scale-tuned (100 / 200 / 500 steps) so total eval cost stays ~10-15% of training time.
- **Batched inference** added (`run_inference_batched`, `eval_batch_size=8`) — 3-4× faster eval throughout (Marco-ASR periodic eval, zero-shot eval, post-FT eval).

**Methodological choices baked in:**

- **Model**: Qwen2-Audio-7B-Instruct (Whisper-LV3 audio encoder + Qwen2-7B LLM, jointly trained, with pooling layer and adapter — see Qwen2-Audio Technical Report).
- **Fine-tuning**: LoRA with `r=32`, `target_modules="all-linear"` (covers encoder + LLM + adapter — verified against PEFT docs; this matches the field convention for Qwen2-Audio FT).
- **Prompt (FT)**: `"Transcribe speech to text "` — exact prompt used in Qwen2-Audio's ASR pretraining. Minimizes distribution shift between pretraining and FT.
- **Prompt (zero-shot reference)**: `"Transcribe speech to text in Moroccan Darija."` — reported as a comparison point only.
- **Adaptive LR**: Marco-ASR Algorithm 1 for LLM-based ASR (Ni et al., Dec 2025, §2.1.2): initial LR scales linearly with WER₀, recalibrated periodically during training, with WER-based early stopping.
- **Evaluation**: corpus-level WER/CER (micro-average) + code-switching partitioning + bootstrap 95% CIs.
- **Normalization**: Casablanca paper scheme (Talafha et al., 2024) — verified independently with edge-case tests before deployment.
- **Dataset**: MoulSot-Full golden subset (`atlasia/MoulSot-Full`, ~80h with code-switching preserved in transcripts).
- **OOD evaluation**: Casablanca Morocco subset (`UBC-NLP/Casablanca`).

**Top-level configuration is in Cell 2** — adjust `MODE`, `DATA_SCALE_HOURS`, and paths there.

**References**: Chu et al. (2024, Qwen2-Audio Tech Report), Talafha et al. (2024, Casablanca), Ni et al. (2025, Marco-ASR), Hu et al. (2021, LoRA), Radford et al. (2023, Whisper).


In [1]:
# === CELL 2: Configuration ===
# All knobs in one place. Edit here, then run the rest of the notebook top-to-bottom.

import os

# --- Mode ---
# fine_tune     : full pipeline (zero-shot eval → WER₀ → fine-tune → final eval)
# zero_shot     : only run the two zero-shot evaluations (no FT)
# wer0_only     : only compute WER₀ and print initial LR (use to plan a run)
# evaluate_only : load a pre-trained adapter and evaluate on test set
MODE = "fine_tune"

# --- Data ---
DATA_SCALE_HOURS = 10                            # 10, 30, or 80
MOULSOT_DATASET_ID = "atlasia/MoulSot-Full"
MOULSOT_CONFIG = "100-gt-2.5"                   # 80h "golden" subset with the publisher's official train/test split
VAL_SET_SIZE = 2000                              # samples carved from publisher's train, used for WER₀ + periodic eval

CASABLANCA_DATASET_ID = "UBC-NLP/Casablanca"
CASABLANCA_SUBSET = "Morocco"                   # for OOD eval slice
CASABLANCA_OOD_MAX_SAMPLES = 500                # cap OOD eval set size

# --- Prompts ---
# DO NOT change TASK_PROMPT_FT — it is Qwen2-Audio's training-time ASR prompt.
TASK_PROMPT_FT = "Transcribe speech to text "
TASK_PROMPT_ZS_DARIJA = "Transcribe speech to text in Moroccan Darija."

# --- Model ---
MODEL_ID = "Qwen/Qwen2-Audio-7B-Instruct"
DEVICE = "cuda"

# --- Robustness / resume ---
USE_FLASH_ATTENTION_2 = True   # tries FA2, falls back to SDPA on import or runtime failure
RESUME_FROM_CHECKPOINT = True  # auto-resume from latest trainer checkpoint if found

# --- LoRA (locked per methodology) ---
LORA_RANK = 32
LORA_ALPHA = 32
LORA_DROPOUT = 0.1
USE_RSLORA = True
LORA_BIAS = "none"

# --- Training ---
PER_DEVICE_BATCH_SIZE = 4                       # was 1; aligns with SpeechLLMs paper (4 per device)
GRADIENT_ACCUMULATION_STEPS = 4                 # effective batch size = 16
NUM_TRAIN_EPOCHS = 3                            # Marco-ASR will likely stop earlier
WEIGHT_DECAY = 0.01
MAX_AUDIO_DURATION_SEC = 30.0                   # Whisper encoder max; longer segments are filtered out

# --- Marco-ASR (Ni et al. 2025, §2.1.2) ---
USE_MARCO_ASR = True
MARCO_LR_MIN = 1e-6                             # η'_min for LLM-based models
MARCO_LR_MAX = 1e-4                             # η'_max for LLM-based models

# Eval subset size scales with training data volume.
# Statistical motivation: σ(corpus WER) ≈ √(p(1-p) / (N * avg_words)) ≈
#   N=50 → σ≈2.0pp,  N=200 → σ≈1.0pp,  N=500 → σ≈0.6pp,  N=1000 → σ≈0.4pp
# At MIN_DELTA=0.5pp, anything below ~500 makes the early-stopping signal noise-dominated.
# The scale-keyed defaults below match measurement precision to data scale.
MARCO_EVAL_SUBSET_BY_SCALE = {10: 200, 20: 300, 30: 500, 80: 1000}
MARCO_EVAL_STEPS_BY_SCALE = {10: 100, 20: 150, 30: 200, 80: 500}
MARCO_EVAL_SUBSET_SIZE = MARCO_EVAL_SUBSET_BY_SCALE.get(DATA_SCALE_HOURS, 200)
MARCO_EVAL_STEPS = MARCO_EVAL_STEPS_BY_SCALE.get(DATA_SCALE_HOURS, 100)
MARCO_EVAL_BATCH_SIZE = 4                       # batched generation. Reduce to 2 if you see CUDA OOM in the audio encoder.

MARCO_WER0_SUBSET_SIZE = 200                    # # utterances for WER₀ (paper recommendation)
MARCO_PATIENCE = 3                              # consecutive non-improving evals → stop
MARCO_MIN_DELTA_PCT = 0.5                       # absolute WER points to count as improvement

# --- Evaluation ---
BOOTSTRAP_RESAMPLES = 1000
BOOTSTRAP_CI = 0.95
TEST_SET_MAX_SAMPLES = None                     # None = use all of test split

# --- Paths (RunPod-friendly defaults) ---
OUTPUT_DIR = "/workspace/outputs"
BEST_CKPT_DIR = os.path.join(OUTPUT_DIR, f"best_marco_asr_{DATA_SCALE_HOURS}h")
LOGS_DIR = os.path.join(OUTPUT_DIR, "logs")
RESULTS_DIR = os.path.join(OUTPUT_DIR, "results")
TRAINER_OUTPUT_DIR = os.path.join(OUTPUT_DIR, f"trainer_{DATA_SCALE_HOURS}h")
EVAL_ADAPTER_PATH = BEST_CKPT_DIR              # used in MODE="evaluate_only"

HUB_MODEL_ID = None                             # e.g. "yourusername/qwen2audio-darija-asr-80h"

# --- Misc ---
SEED = 42
DEBUG = False

for d in [OUTPUT_DIR, BEST_CKPT_DIR, LOGS_DIR, RESULTS_DIR, TRAINER_OUTPUT_DIR]:
    os.makedirs(d, exist_ok=True)

print("Configuration loaded:")
print(f"  Mode:                 {MODE}")
print(f"  Data scale:           {DATA_SCALE_HOURS}h")
print(f"  LoRA rank:            {LORA_RANK} (alpha={LORA_ALPHA}, rsLoRA={USE_RSLORA})")
print(f"  Per-device batch:     {PER_DEVICE_BATCH_SIZE}")
print(f"  Grad accumulation:    {GRADIENT_ACCUMULATION_STEPS}  (effective batch = {PER_DEVICE_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS})")
print(f"  Marco-ASR enabled:    {USE_MARCO_ASR}")
print(f"  Marco eval subset:    {MARCO_EVAL_SUBSET_SIZE} utt (scale-tuned for σ ≤ MIN_DELTA)")
print(f"  Marco eval interval:  every {MARCO_EVAL_STEPS} steps")
print(f"  Marco eval batch:     {MARCO_EVAL_BATCH_SIZE} (batched generation for speed)")
print(f"  Output dir:           {OUTPUT_DIR}")
print(f"  Flash Attention 2:    {USE_FLASH_ATTENTION_2}")
print(f"  Auto-resume:          {RESUME_FROM_CHECKPOINT}")


Configuration loaded:
  Mode:                 fine_tune
  Data scale:           10h
  LoRA rank:            32 (alpha=32, rsLoRA=True)
  Per-device batch:     4
  Grad accumulation:    4  (effective batch = 16)
  Marco-ASR enabled:    True
  Marco eval subset:    200 utt (scale-tuned for σ ≤ MIN_DELTA)
  Marco eval interval:  every 100 steps
  Marco eval batch:     4 (batched generation for speed)
  Output dir:           /workspace/outputs
  Flash Attention 2:    True
  Auto-resume:          True


## 1. Environment setup

In [1]:
# === CELL 3a : Install dependencies ===
# Uncomment if running on a fresh RunPod instance.
# ===========================================================================
# ===========================================================================
# Run these in terminal on RunPod, or uncomment for notebook execution:
#
!python -m pip install --upgrade pip -q
!pip install transformers==4.46.3
!pip install -qU accelerate datasets bitsandbytes hf_transfer \
     tensorboard librosa jiwer regex
!pip install torchcodec soundfile
!pip install peft==0.13.2
!pip uninstall -y torchcodec
!pip install -q flash-attn --no-build-isolation  # ~30s with prebuilt wheel; up to 10min if compiled
!pip install "datasets<3.0" soundfile
!apt-get update -qq && apt-get install -y -qq libsndfile1 > /dev/null 2>&1

  Using cached transformers-4.46.3-py3-none-any.whl.metadata (44 kB)
  Using cached huggingface_hub-0.36.2-py3-none-any.whl.metadata (15 kB)
  Using cached regex-2026.5.9-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (40 kB)
  Using cached tokenizers-0.20.3-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.7 kB)
  Using cached safetensors-0.7.0-cp38-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.1 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached hf_xet-1.5.0-cp37-abi3-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (4.9 kB)
Using cached transformers-4.46.3-py3-none-any.whl (10.0 MB)
Using cached huggingface_hub-0.36.2-py3-none-any.whl (566 kB)
Using cached hf_xet-1.5.0-cp37-abi3-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (4.5 MB)
Using cached tokenizers-0.20.3-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (3.0 MB)
Using cached regex-2026.5.9-cp312-cp

In [2]:
# === CELL 3b: Imports ===
import os
import json
import time
import gc
import random
from typing import Dict, List, Tuple, Optional, Callable, Any

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import numpy as np
import torch
from torch.utils.data import DataLoader

import re
import librosa
from datasets import load_dataset, Dataset

from transformers import (
    Qwen2AudioForConditionalGeneration,
    AutoProcessor,
    TrainingArguments,
    Trainer,
    TrainerCallback,
    TrainerControl,
    TrainerState,
)
from peft import LoraConfig, get_peft_model, PeftModel

from jiwer import wer as jiwer_wer, cer as jiwer_cer

# Reproducibility
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Speed up HF downloads on RunPod
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


PyTorch: 2.8.0+cu128
CUDA available: True
Device: NVIDIA A40
Memory: 47.7 GB


In [3]:
# === CELL 3c: HF Hub login ===
# Set HF_TOKEN as an env var on RunPod, OR call login() interactively.
from huggingface_hub import login
hf_token = os.environ.get("HF_TOKEN")
if hf_token:
    login(token=hf_token)
    print("Logged in to HF Hub via HF_TOKEN env var.")
else:
    print("HF_TOKEN not set. Call login() interactively if needed:")
    print("  >>> from huggingface_hub import login; login()")


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Logged in to HF Hub via HF_TOKEN env var.


## 2. Model and processor loading

Loads Qwen2-Audio-7B-Instruct in bfloat16 (native A100/A40/L40 precision). On smaller GPUs, switch `TORCH_DTYPE` to `torch.float16` or enable 4-bit quantization.

In [4]:
# === CELL 4: Load Qwen2-Audio model and processor ===
TORCH_DTYPE = torch.bfloat16

def _load_qwen2audio_with_fa2_fallback():
    """Try Flash Attention 2 → fall back to SDPA. Catches both ImportError and runtime errors."""
    #if USE_FLASH_ATTENTION_2:
     #   try:
      #      import flash_attn
       #     try:
        #        m = Qwen2AudioForConditionalGeneration.from_pretrained(
         #           MODEL_ID,
          #          torch_dtype=TORCH_DTYPE,
           #         attn_implementation="flash_attention_2",
            #        trust_remote_code=True,
             #   ).to(DEVICE)
              #  print(f"Loaded with Flash Attention 2 (flash_attn v{flash_attn.__version__})")
               # return m, "flash_attention_2"
           # except Exception as e:
           #     print(f"FA2 load failed at runtime: {type(e).__name__}: {e}")
           #     print("Falling back to SDPA...")
      #  except ImportError:
    print("flash_attn not installed; falling back to SDPA")
    m = Qwen2AudioForConditionalGeneration.from_pretrained(
        MODEL_ID,
        torch_dtype=TORCH_DTYPE,
        attn_implementation="sdpa",
        trust_remote_code=True,
    ).to(DEVICE)
    print("Loaded with SDPA (PyTorch native scaled dot-product attention)")
    return m, "sdpa"

print(f"Loading {MODEL_ID} in {TORCH_DTYPE}...")
model, attn_impl_used = _load_qwen2audio_with_fa2_fallback()
model.eval()  # default to eval; we switch to train() before training

processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)

target_sr = processor.feature_extractor.sampling_rate
print(f"Model loaded on {DEVICE}, dtype={TORCH_DTYPE}, attention={attn_impl_used}")
print(f"Audio target sampling rate: {target_sr} Hz")
print(f"Whisper encoder n_samples: {processor.feature_extractor.n_samples} ({processor.feature_extractor.n_samples / target_sr:.1f}s max)")

The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.


Loading Qwen/Qwen2-Audio-7B-Instruct in torch.bfloat16...
flash_attn not installed; falling back to SDPA


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

Loaded with SDPA (PyTorch native scaled dot-product attention)
Model loaded on cuda, dtype=torch.bfloat16, attention=sdpa
Audio target sampling rate: 16000 Hz
Whisper encoder n_samples: 480000 (30.0s max)


## 3. Dataset loading

### 3.1 MoulSot-Full

The `atlasia/MoulSot-Full` dataset contains ~1500 hours of Darija audio, with an 80-hour "golden" subset of high-quality transcriptions. We use the golden subset for both training and primary in-distribution evaluation.

**On the data scales**: methodology says "10h / 30h / 80h fine-tuning". After the 80/10/10 train/val/test split, the actual maximum trainable hours is ~64h (80% of 80h). So:
- `DATA_SCALE_HOURS=10` → 10h subset of the train pool
- `DATA_SCALE_HOURS=30` → 30h subset of the train pool
- `DATA_SCALE_HOURS=80` → use the full train pool (~64h actually)

The 10h ⊂ 30h ⊂ 80h nesting is preserved by using a fixed shuffle seed.

In [5]:
# === CELL 5: Load MoulSot-Full ===
print(f"Loading {MOULSOT_DATASET_ID} (config: {MOULSOT_CONFIG})...")
ds_train_full = load_dataset(MOULSOT_DATASET_ID, MOULSOT_CONFIG, split="train")
ds_test_official = load_dataset(MOULSOT_DATASET_ID, MOULSOT_CONFIG, split="test")

print(f"Loaded train: {len(ds_train_full)} samples")
print(f"Loaded test:  {len(ds_test_official)} samples (publisher's official held-out split)")
print(f"Columns: {ds_train_full.column_names}")
print(f"Features: {ds_train_full.features}")
print()
print("First train sample (showing keys, hiding audio array):")
sample = ds_train_full[0]
for k, v in sample.items():
    if k == 'audio':
        print(f"  {k}: array shape={v['array'].shape}, sr={v['sampling_rate']}")
    else:
        v_str = str(v)
        print(f"  {k}: {v_str[:200]}")


Loading atlasia/MoulSot-Full (config: 100-gt-2.5)...


Resolving data files:   0%|          | 0/24 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/24 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/21 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/24 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/24 [00:00<?, ?it/s]

Loaded train: 79857 samples
Loaded test:  1993 samples (publisher's official held-out split)
Columns: ['id', 'audio', 'sample_rate', 'n_channels', 'pesq_hyp', 'stoi_hyp', 'si_sdr_hyp', 'text', 'channel', 'duration']
Features: {'id': Value(dtype='string', id=None), 'audio': Audio(sampling_rate=None, mono=True, decode=True, id=None), 'sample_rate': Value(dtype='int64', id=None), 'n_channels': Value(dtype='int64', id=None), 'pesq_hyp': Value(dtype='float64', id=None), 'stoi_hyp': Value(dtype='float64', id=None), 'si_sdr_hyp': Value(dtype='float64', id=None), 'text': Value(dtype='string', id=None), 'channel': Value(dtype='string', id=None), 'duration': Value(dtype='float64', id=None)}

First train sample (showing keys, hiding audio array):
  id: @HichamNostikOfficial_89_115
  audio: array shape=(30400,), sr=16000
  sample_rate: 16000
  n_channels: 1
  pesq_hyp: 3.33984375
  stoi_hyp: 0.99658203125
  si_sdr_hyp: 26.8125
  text: يحتوي هاد الاحاسيس كاملة
  channel: @HichamNostikOfficial
  dur

### 3.2 Subset selection by hours

We select samples up to a target duration. With the same shuffle seed, smaller subsets are guaranteed to be subsets of larger ones (10h ⊂ 30h ⊂ full).

In [6]:
# === CELL 6: Subset selection helper ===
def select_by_hours(dataset, target_hours, seed=SEED, duration_field='duration'):
    """Select samples from the dataset until cumulative duration reaches target_hours.

    Uses a fixed RNG seed so the order is deterministic — meaning select_by_hours(ds, 10)
    is a strict prefix of select_by_hours(ds, 30) under the same seed.
    """
    target_seconds = target_hours * 3600

    # Try to use a pre-computed duration column for speed (avoids loading audio arrays).
    # If absent, fall back to computing from audio. The fallback is slow on large datasets.
    durations = None
    try:
        if duration_field in dataset.column_names:
            durations = dataset[duration_field]
    except Exception:
        pass

    rng = np.random.RandomState(seed)
    indices = np.arange(len(dataset))
    rng.shuffle(indices)

    selected = []
    cumulative = 0.0
    for i in indices:
        if durations is not None:
            dur = float(durations[int(i)])
        else:
            ex = dataset[int(i)]
            audio = ex['audio']
            dur = len(audio['array']) / audio['sampling_rate']
        cumulative += dur
        selected.append(int(i))
        if cumulative >= target_seconds:
            break

    return dataset.select(selected), cumulative / 3600.0


def filter_too_long(dataset, max_seconds=MAX_AUDIO_DURATION_SEC, duration_field='duration'):
    """Drop samples longer than max_seconds (Whisper encoder cap = 30s)."""
    if duration_field in dataset.column_names:
        before = len(dataset)
        ds = dataset.filter(lambda ex: ex[duration_field] <= max_seconds)
        print(f"  Filtered out {before - len(ds)} samples >{max_seconds}s "
              f"(kept {len(ds)}/{before})")
        return ds
    # Fallback: compute on the fly
    def keep(ex):
        a = ex['audio']
        return len(a['array']) / a['sampling_rate'] <= max_seconds
    before = len(dataset)
    ds = dataset.filter(keep)
    print(f"  Filtered out {before - len(ds)} samples >{max_seconds}s "
          f"(kept {len(ds)}/{before})")
    return ds


### 3.3 Build train / val / test / WER₀ / periodic-eval splits

- **train**: subset of the train pool to `DATA_SCALE_HOURS` hours.
- **val**: fixed 10% of the golden subset, used for periodic eval and WER₀ during training.
- **test**: fixed 10% of the golden subset, held out for final reporting.
- **WER₀ subset**: 200 utterances drawn from `val` (separate seed).
- **periodic eval subset**: 50 utterances drawn from `val`, disjoint from the WER₀ subset.

These last two are kept *fixed across all evaluation runs* so the metric trajectory is comparable step-to-step.

In [7]:
# === CELL 7: Build all splits ===

# Filter overly long audio (>30s) — Whisper encoder cannot ingest these
print("Filtering out audio segments >30s in train pool...")
ds_train_clean = filter_too_long(ds_train_full, max_seconds=MAX_AUDIO_DURATION_SEC)
print("Filtering out audio segments >30s in test set...")
ds_test_clean = filter_too_long(ds_test_official, max_seconds=MAX_AUDIO_DURATION_SEC)

# Test set = publisher's official held-out split. Never touched during training.
test_set = ds_test_clean

# Validation set = VAL_SET_SIZE samples carved from publisher's train (seed=42 → deterministic).
# Used for WER₀ + Marco-ASR periodic eval. Disjoint from training pool by HF's train_test_split contract.
print(f"\nCarving {VAL_SET_SIZE}-sample validation set from publisher's train (seed={SEED})...")
splits = ds_train_clean.train_test_split(test_size=VAL_SET_SIZE, seed=SEED)
train_pool = splits['train']
val_set = splits['test']

print(f"  train_pool: {len(train_pool)} samples (publisher's train minus our val carve)")
print(f"  val_set:    {len(val_set)} samples (carved from publisher's train, used for WER₀ + periodic eval)")
print(f"  test_set:   {len(test_set)} samples (publisher's official test — held out)")

# Disjointness sanity (cheap to verify): val_set and train_pool can't overlap because
# they come from the same train_test_split call. test_set is the publisher's own split,
# disjoint from publisher's train by their construction.
# Subset train_pool to the target scale (deterministically nested)
print(f"\nSelecting {DATA_SCALE_HOURS}h training subset...")
train_set, actual_train_hours = select_by_hours(
    train_pool, target_hours=DATA_SCALE_HOURS, seed=SEED
)
if actual_train_hours < DATA_SCALE_HOURS * 0.95:
    print(f"  WARNING: requested {DATA_SCALE_HOURS}h but only {actual_train_hours:.2f}h available "
          f"(train_pool is {sum(train_pool['duration']) / 3600 if 'duration' in train_pool.column_names else '~64'}h max)")
print(f"  train_set: {len(train_set)} samples, ~{actual_train_hours:.2f}h actual")

# WER₀ + periodic eval subsets — both drawn from val_set, disjoint
val_indices = list(range(len(val_set)))
rng_eval = np.random.RandomState(SEED + 1)  # separate seed from main split
rng_eval.shuffle(val_indices)

n_wer0 = min(MARCO_WER0_SUBSET_SIZE, len(val_set))
n_peval = min(MARCO_EVAL_SUBSET_SIZE, len(val_set) - n_wer0)

wer0_indices = sorted(val_indices[:n_wer0])
peval_indices = sorted(val_indices[n_wer0:n_wer0 + n_peval])

wer0_subset = val_set.select(wer0_indices)
periodic_eval_subset = val_set.select(peval_indices)

print(f"  wer0_subset:           {len(wer0_subset)} samples")
print(f"  periodic_eval_subset:  {len(periodic_eval_subset)} samples")

# Sanity: must be disjoint
assert set(wer0_indices).isdisjoint(set(peval_indices)), "WER₀ and periodic eval subsets overlap!"


Filtering out audio segments >30s in train pool...
  Filtered out 0 samples >30.0s (kept 79857/79857)
Filtering out audio segments >30s in test set...
  Filtered out 0 samples >30.0s (kept 1993/1993)

Carving 2000-sample validation set from publisher's train (seed=42)...
  train_pool: 77857 samples (publisher's train minus our val carve)
  val_set:    2000 samples (carved from publisher's train, used for WER₀ + periodic eval)
  test_set:   1993 samples (publisher's official test — held out)

Selecting 10h training subset...
  train_set: 7899 samples, ~10.00h actual
  wer0_subset:           200 samples
  periodic_eval_subset:  200 samples


### 3.4 Casablanca Morocco subset for OOD evaluation

Used only for evaluation, never for training. Allows comparison with published numbers in the Casablanca paper.

In [8]:
# === CELL 8: Load Casablanca OOD eval slice ===
print(f"Loading {CASABLANCA_DATASET_ID} ({CASABLANCA_SUBSET})...")
casa_ds = load_dataset(CASABLANCA_DATASET_ID, CASABLANCA_SUBSET)
print(f"Available splits: {list(casa_ds.keys())}")
for sp, ds_sp in casa_ds.items():
    print(f"  {sp}: {len(ds_sp)} samples")

# Use the test split if available, else validation. Cap to CASABLANCA_OOD_MAX_SAMPLES.
if 'test' in casa_ds:
    casa_eval = casa_ds['test']
elif 'validation' in casa_ds:
    casa_eval = casa_ds['validation']
else:
    casa_eval = list(casa_ds.values())[0]

if CASABLANCA_OOD_MAX_SAMPLES is not None and len(casa_eval) > CASABLANCA_OOD_MAX_SAMPLES:
    casa_eval = casa_eval.select(range(CASABLANCA_OOD_MAX_SAMPLES))

print(f"\nCasablanca OOD eval set: {len(casa_eval)} samples")
print(f"Columns: {casa_eval.column_names}")


Loading UBC-NLP/Casablanca (Morocco)...
Available splits: ['validation', 'test']
  validation: 1045 samples
  test: 1045 samples

Casablanca OOD eval set: 500 samples
Columns: ['audio', 'seg_id', 'transcription', 'gender', 'duration']


## 4. Evaluation utilities

### 4.1 Casablanca normalization

Following Talafha et al. (2024). Applied to BOTH reference and hypothesis before WER/CER.

In [9]:
# === CELL 9: Casablanca normalization ===
# Verified independently with edge-case tests.
_DIACRITICS_RE = re.compile(
    r"[\u0610-\u061A\u064B-\u065F\u0670\u06D6-\u06DC\u06DF-\u06E8\u06EA-\u06ED]"
)
_HAMZA_MADDA_RE = re.compile(r"[إأآٱ]")
_PUNCT_RE = re.compile(r"[^\w\s%@]")
_WS_RE = re.compile(r"\s+")
_EASTERN_TO_WESTERN = str.maketrans("٠١٢٣٤٥٦٧٨٩", "0123456789")


def casablanca_normalize(text: str) -> str:
    """Casablanca paper normalization (Talafha et al., 2024).

    Steps:
      1. Remove diacritics, hamzas, maddas (keep bare alif).
      2. Convert Eastern → Western Arabic numerals.
      3. Lowercase Latin characters (for fair WER on code-switched segments).
      4. Remove all punctuation EXCEPT % and @.
      5. Normalize whitespace.
    """
    if not text:
        return ""
    text = _DIACRITICS_RE.sub("", text)
    text = _HAMZA_MADDA_RE.sub("ا", text)
    text = text.translate(_EASTERN_TO_WESTERN)
    text = text.lower()
    text = _PUNCT_RE.sub("", text)
    text = _WS_RE.sub(" ", text).strip()
    return text


# Unit tests (run on first execution, fast)
_test_cases = [
    ("بِسْمِ اللَّهِ", "بسم الله"),
    ("أنا إنسان آخر", "انا انسان اخر"),
    ("٩٢ دينار", "92 دينار"),
    ("اشتريت l'Voiture", "اشتريت lvoiture"),
    ("J'AI MANGÉ", "jai mangé"),
    ("", ""),
]
for raw, expected in _test_cases:
    got = casablanca_normalize(raw)
    assert got == expected, f"Normalization failed: '{raw}' → '{got}', expected '{expected}'"
print("Casablanca normalization: 6/6 tests passed ✓")


Casablanca normalization: 6/6 tests passed ✓


In [10]:
# === CELL 9b: Qwen2-Audio boilerplate stripping ===
# Applied to model PREDICTIONS only, BEFORE Casablanca normalization.
# Rationale: Qwen2-Audio is instruction-tuned and frequently emits conversational
# framing like 'The transcription of the speech is: "..."' before the actual transcript.
# Treating that framing as part of the prediction inflates WER by tens of percentage
# points via spurious insertions. Whisper produces raw transcripts natively, so this
# strip aligns the two outputs for a fair WER comparison.
# (See methodology: this is documented as part of the eval pipeline, with raw_pred
# preserved in the detail dump so the boilerplate is auditable.)

_BOILERPLATE_PATTERNS = [
    r"the\s+transcription\s+of\s+(?:the\s+|this\s+)?(?:speech|audio|recording|sound|conversation)?\s*(?:is|reads|says|goes)?\s*[:\.\-,]?\s*",
    r"the\s+transcription\s*(?:is|reads)?\s*[:\.\-,]?\s*",
    r"here\s*(?:'s|is)\s+(?:the\s+)?transcription\s*[:\.\-,]?\s*",
    r"the\s+(?:speech|audio|recording)\s+(?:in\s+(?:the\s+)?(?:audio|recording)\s+)?(?:is\s+)?(?:transcribed\s+as|reads|says|goes|states)\s*[:\.\-,]?\s*",
    r"the\s+(?:audio|speech|recording|speaker|narrator|person)\s+(?:says|reads|goes|states|tells|mentions)\s*[:\.\-,]?\s*",
    r"transcription\s*[:\.\-,]?\s*",
    r"i\s+(?:can\s+)?(?:heard?|hear)\s*[:\.\-,]?\s*",
    r"it\s+(?:says|reads|goes)\s*[:\.\-,]?\s*",
    r"the\s+(?:text|transcript)\s+(?:is|reads|says)\s*[:\.\-,]?\s*",
    r"(?:based\s+on|according\s+to)\s+(?:the\s+)?audio\s*[,\.]?\s*",
    r"in\s+(?:this|the)\s+audio\s*[,\.]?\s*",
]
_BOILERPLATE_RE = re.compile(
    r"^\s*(?:" + "|".join(_BOILERPLATE_PATTERNS) + r")",
    re.IGNORECASE,
)
_LEADING_WRAPPERS_RE = re.compile(r'^[\s"\'\u201c\u201d\u2018\u2019\(\[\{`]+')
_TRAILING_WRAPPERS_RE = re.compile(r'[\s"\'\u201c\u201d\u2018\u2019\)\]\}\.`]+$')


def strip_qwen_boilerplate(text: str) -> str:
    """Strip Qwen2-Audio conversational ASR boilerplate. See Cell 9b docstring."""
    if not text:
        return text
    # Order matters: strip leading wrappers first so the prefix matcher
    # can see underneath things like '"The transcription is: hello"'
    text = _LEADING_WRAPPERS_RE.sub('', text)
    # Iteratively strip (handles rare nested patterns)
    for _ in range(3):
        new = _BOILERPLATE_RE.sub('', text)
        if new == text:
            break
        text = new
    # Strip newly-exposed leading wrappers + trailing wrappers/period
    text = _LEADING_WRAPPERS_RE.sub('', text)
    text = _TRAILING_WRAPPERS_RE.sub('', text)
    return text.strip()


# Sanity tests on representative inputs (will print PASS/FAIL on first run)
_BOILERPLATE_TESTS = [
    ('The transcription of the speech is: "زورا قديما".', 'زورا قديما'),
    ('The transcription is: hello world', 'hello world'),
    ("Here's the transcription: مرحبا", 'مرحبا'),
    ('The speech in the audio is transcribed as: مرحبا', 'مرحبا'),
    ('The audio says hello', 'hello'),
    ('It says: مرحبا', 'مرحبا'),
    ('مرحبا بك', 'مرحبا بك'),                              # no prefix — unchanged
    ('hello world', 'hello world'),                        # no prefix — unchanged
    ('The man went to the store', 'The man went to the store'),  # "The X" but not a transcription prefix
    ('', ''),                                              # empty
    ('"The transcription is: hello"', 'hello'),            # wrapped + prefix
    ('THE TRANSCRIPTION IS: HELLO', 'HELLO'),              # case-insensitive
]
_failed = []
for _raw, _expected in _BOILERPLATE_TESTS:
    _got = strip_qwen_boilerplate(_raw)
    if _got != _expected:
        _failed.append((_raw, _expected, _got))
if _failed:
    for _raw, _exp, _got in _failed:
        print(f"  FAIL: {_raw!r} → expected {_exp!r}, got {_got!r}")
    raise AssertionError(f"Boilerplate-strip failed {len(_failed)}/{len(_BOILERPLATE_TESTS)} tests")
print(f"Qwen2-Audio boilerplate strip: {len(_BOILERPLATE_TESTS)}/{len(_BOILERPLATE_TESTS)} tests passed ✓")

Qwen2-Audio boilerplate strip: 12/12 tests passed ✓


### 4.2 Code-switching detection

A reference is classified as code-switched iff it contains BOTH an Arabic character AND a Latin character. Digits and punctuation are ignored by virtue of the Unicode-range checks (they don't match either class).

In [11]:
# === CELL 10: Code-switching detection ===
_ARABIC_RANGE_RE = re.compile(r'[\u0600-\u06FF]')
_LATIN_RANGE_RE = re.compile(r'[A-Za-z]')


def is_code_switched(raw_reference: str) -> bool:
    """True iff reference contains both Arabic AND Latin characters.

    Pure-Latin segments (e.g., entirely French) → False (monolingual class).
    Pure-Arabic segments (Darija/MSA) → False (monolingual class).
    Mixed → True (code-switched class).
    """
    if not raw_reference:
        return False
    has_arabic = bool(_ARABIC_RANGE_RE.search(raw_reference))
    has_latin = bool(_LATIN_RANGE_RE.search(raw_reference))
    return has_arabic and has_latin


# Sanity tests
assert not is_code_switched("بسم الله")
assert not is_code_switched("hello world")
assert is_code_switched("اشتريت l'voiture")
assert not is_code_switched("")
assert not is_code_switched("١٢٣ 456 ١٢٣")  # only digits, no letters
print("Code-switching detection: 5/5 tests passed ✓")


Code-switching detection: 5/5 tests passed ✓


### 4.3 Inference function

Single-utterance inference. The model is passed in explicitly (not captured globally) so it works for both the base model (zero-shot eval) and the LoRA-wrapped model (post-FT eval) without modification.

In [12]:
# === CELL 11: Inference function (with timing) ===
def run_inference(
    audio_array: np.ndarray,
    sampling_rate: int,
    prompt: str,
    model,
    processor,
    device: str = DEVICE,
    max_new_tokens: int = 256,
) -> str:
    """Run Qwen2-Audio inference on a single audio sample.

    Returns the decoded response (string). Caller is responsible for timing
    around this call (use torch.cuda.synchronize() before measuring).
    """
    target_sr = processor.feature_extractor.sampling_rate
    audio_array = audio_array.astype(np.float32, copy=False)
    if sampling_rate != target_sr:
        audio_array = librosa.resample(audio_array, orig_sr=sampling_rate, target_sr=target_sr)

    conversation = [
        {
            "role": "user",
            "content": [
                {"type": "audio", "audio_url": "placeholder"},
                {"type": "text", "text": prompt},
            ],
        }
    ]
    text = processor.apply_chat_template(
        conversation, add_generation_prompt=True, tokenize=False
    )

    inputs = processor(
        text=text,
        audios=[audio_array],
        sampling_rate=target_sr,
        return_tensors="pt",
        padding=True,
    ).to(device)

    # FIX 1: Ensure pad_token_id exists to prevent generation warnings/errors
    if processor.tokenizer.pad_token_id is None:
        processor.tokenizer.pad_token_id = processor.tokenizer.eos_token_id

    with torch.no_grad():
        generate_ids = model.generate(**inputs, max_new_tokens=max_new_tokens, 
                       pad_token_id=processor.tokenizer.pad_token_id # <--- ADDED HERE ai studio
                       )

    # Slice off the prompt tokens, decode only the model's response
    generate_ids = generate_ids[:, inputs.input_ids.size(1):]
    response = processor.batch_decode(
        generate_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )[0]
    return response.strip()


def run_inference_batched(
    audios_with_sr: List[Tuple[np.ndarray, int]],
    prompt: str,
    model,
    processor,
    device: str = DEVICE,
    max_new_tokens: int = 256,
) -> List[str]:
    """Batched Qwen2-Audio inference. Returns list of decoded transcripts (one per input).

    Sets tokenizer padding_side='left' for the call (required for batched generation
    so all sequences end at the same position before generation begins), then restores it.
    """
    if not audios_with_sr:
        return []
    target_sr = processor.feature_extractor.sampling_rate
    audios = []
    for audio, sr in audios_with_sr:
        a = audio.astype(np.float32, copy=False)
        if sr != target_sr:
            a = librosa.resample(a, orig_sr=sr, target_sr=target_sr)
        audios.append(a)

    conversations = [
        [{"role": "user", "content": [
            {"type": "audio", "audio_url": "placeholder"},
            {"type": "text", "text": prompt},
        ]}]
        for _ in audios
    ]
    texts = [
        processor.apply_chat_template(conv, add_generation_prompt=True, tokenize=False)
        for conv in conversations
    ]

    # Left-padding is required for batched autoregressive generation: all input
    # sequences must end at the same position so generation continues uniformly.
    old_side = processor.tokenizer.padding_side
    processor.tokenizer.padding_side = "left"
    try:
        inputs = processor(
            text=texts,
            audios=audios,
            sampling_rate=target_sr,
            return_tensors="pt",
            padding=True,
        ).to(device)
    finally:
        processor.tokenizer.padding_side = old_side
        
    # FIX 1: Ensure pad_token_id exists to prevent generation warnings/errors by aistudio
    if processor.tokenizer.pad_token_id is None:
        processor.tokenizer.pad_token_id = processor.tokenizer.eos_token_id

    with torch.no_grad():
        generate_ids = model.generate(**inputs, max_new_tokens=max_new_tokens,
                    pad_token_id=processor.tokenizer.pad_token_id # <--- ADDED HERE ai studio
                    )

    # With left-padding, generated tokens start at column inputs.input_ids.size(1) for all rows
    input_len = inputs.input_ids.size(1)
    generate_ids = generate_ids[:, input_len:]
    responses = processor.batch_decode(
        generate_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )
    return [r.strip() for r in responses]


### 4.4 Bootstrap confidence intervals

For statistical significance per methodology §5.2. Resamples utterance-level pairs with replacement, recomputes the metric on each resample, returns mean ± 95% CI.

In [13]:
# === CELL 12: Bootstrap CI ===
def bootstrap_metric(
    pairs: List[Tuple[str, str]],
    metric_fn: Callable[[List[str], List[str]], float],
    n_resamples: int = BOOTSTRAP_RESAMPLES,
    ci: float = BOOTSTRAP_CI,
    seed: int = SEED,
) -> Optional[Dict[str, float]]:
    """Bootstrap a corpus-level metric.

    pairs:      list of (normalized_ref, normalized_hyp) tuples
    metric_fn:  callable like jiwer_wer / jiwer_cer that takes (refs_list, hyps_list)
                and returns a scalar.
    """
    if not pairs:
        return None
    rng = np.random.RandomState(seed)
    n = len(pairs)
    refs = [p[0] for p in pairs]
    hyps = [p[1] for p in pairs]

    boot_values = []
    for _ in range(n_resamples):
        idx = rng.choice(n, size=n, replace=True)
        b_refs = [refs[i] for i in idx]
        b_hyps = [hyps[i] for i in idx]
        try:
            boot_values.append(metric_fn(b_refs, b_hyps))
        except Exception:
            continue

    if not boot_values:
        return None

    arr = np.array(boot_values)
    alpha = (1.0 - ci) / 2.0
    return {
        "mean": float(arr.mean()),
        "std": float(arr.std()),
        "ci_lower": float(np.percentile(arr, alpha * 100)),
        "ci_upper": float(np.percentile(arr, (1 - alpha) * 100)),
        "n_resamples": int(len(arr)),
    }


def paired_bootstrap_diff(
    pairs_a: List[Tuple[str, str]],
    pairs_b: List[Tuple[str, str]],
    metric_fn: Callable[[List[str], List[str]], float],
    n_resamples: int = BOOTSTRAP_RESAMPLES,
    ci: float = BOOTSTRAP_CI,
    seed: int = SEED,
) -> Optional[Dict[str, float]]:
    """Paired bootstrap of (metric(B) - metric(A)) using same indices for both.

    Both pairs lists must have the same length and same ordering — i.e.,
    pairs_a[i] and pairs_b[i] correspond to the same audio utterance, evaluated
    by two different models.
    """
    assert len(pairs_a) == len(pairs_b), "Paired bootstrap requires same length"
    if not pairs_a:
        return None
    rng = np.random.RandomState(seed)
    n = len(pairs_a)

    refs_a = [p[0] for p in pairs_a]
    hyps_a = [p[1] for p in pairs_a]
    refs_b = [p[0] for p in pairs_b]
    hyps_b = [p[1] for p in pairs_b]

    diffs = []
    for _ in range(n_resamples):
        idx = rng.choice(n, size=n, replace=True)
        try:
            m_a = metric_fn([refs_a[i] for i in idx], [hyps_a[i] for i in idx])
            m_b = metric_fn([refs_b[i] for i in idx], [hyps_b[i] for i in idx])
            diffs.append(m_b - m_a)
        except Exception:
            continue

    if not diffs:
        return None

    arr = np.array(diffs)
    alpha = (1.0 - ci) / 2.0
    return {
        "mean_diff": float(arr.mean()),
        "ci_lower": float(np.percentile(arr, alpha * 100)),
        "ci_upper": float(np.percentile(arr, (1 - alpha) * 100)),
        "n_resamples": int(len(arr)),
        "significant": (np.percentile(arr, alpha * 100) > 0) or (np.percentile(arr, (1 - alpha) * 100) < 0),
    }


### 4.5 Evaluation function

Implements the algorithm: for each sample, partition by ground-truth code-switching status, normalize, and accumulate. Returns corpus-level WER/CER for **full / code-switched / monolingual** subsets, with bootstrap CIs and per-utterance timing.

In [14]:
# === CELL 13: Evaluation with CS partitioning + bootstrap ===
def evaluate_with_partitioning(
    dataset,
    model,
    processor,
    prompt: str,
    *,
    transcription_field: str = 'transcription',
    device: str = DEVICE,
    max_samples: Optional[int] = None,
    eval_batch_size: int = 1,
    do_bootstrap: bool = True,
    verbose: bool = False,
) -> Dict[str, Any]:
    """Evaluate model on dataset with code-switching partitioning.

    Implements the corpus-level WER/CER (micro-average) algorithm with three
    partitions: full, code_switched, monolingual.

    eval_batch_size > 1 uses batched generation (3-4× faster on Qwen2-Audio).
    Per-utterance timing is approximated as batch_time / batch_size in that case;
    set eval_batch_size=1 if you need exact per-utterance timing.
    """
    detail = {"full": [], "code_switched": [], "monolingual": [], "errors": [], "timings": []}

    n_samples = min(max_samples, len(dataset)) if max_samples else len(dataset)

    was_training = model.training
    model.eval()

    # Discover transcription field if needed
    cols = dataset.column_names
    if transcription_field not in cols:
        for fallback in ['transcription', 'text', 'transcript', 'sentence']:
            if fallback in cols:
                transcription_field = fallback
                break
        else:
            raise ValueError(f"No transcription field in dataset; available: {cols}")

    print(f"Evaluating {n_samples} samples with prompt='{prompt}' "
          f"(eval_batch_size={eval_batch_size})")
    print(f"Transcription field: '{transcription_field}'")

    t_eval_start = time.perf_counter()

    for batch_start in range(0, n_samples, eval_batch_size):
        batch_end = min(batch_start + eval_batch_size, n_samples)
        batch_indices = list(range(batch_start, batch_end))

        # Gather batch examples (skip examples that fail to load)
        batch_audios_with_sr = []
        batch_raw_refs = []
        batch_durations = []
        kept_indices = []
        for i in batch_indices:
            try:
                ex = dataset[i]
                audio = ex['audio']['array'].astype(np.float32, copy=False)
                sr = ex['audio']['sampling_rate']
                raw_ref = ex.get(transcription_field, "") or ""
                batch_audios_with_sr.append((audio, sr))
                batch_raw_refs.append(raw_ref)
                batch_durations.append(len(audio) / sr)
                kept_indices.append(i)
            except Exception as e:
                detail["errors"].append({"idx": i, "error": str(e)})
                if verbose:
                    print(f"  [LOAD-ERR] sample {i}: {e}")

        if not kept_indices:
            continue

        # Batched inference with timing
        try:
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            t_start = time.perf_counter()
            with torch.no_grad():
                if eval_batch_size == 1:
                    audio, sr = batch_audios_with_sr[0]
                    raw_preds = [run_inference(audio, sr, prompt, model=model, processor=processor, device=device)]
                else:
                    raw_preds = run_inference_batched(
                        batch_audios_with_sr, prompt, model=model, processor=processor,
                        device=device,
                    )
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            batch_time = time.perf_counter() - t_start
        except Exception as e:
            for i in kept_indices:
                detail["errors"].append({"idx": i, "error": f"inference: {e}"})
            if verbose:
                print(f"  [INFER-ERR] batch starting at {batch_start}: {e}")
            continue

        # Per-utterance time = batch_time / batch_size (approximate)
        per_utt_time = batch_time / len(kept_indices)

        # Process each example in the batch
        for k, (i, raw_ref, raw_pred, audio_dur) in enumerate(
            zip(kept_indices, batch_raw_refs, raw_preds, batch_durations)
        ):
            is_cs = is_code_switched(raw_ref)
            timing = {
                "idx": i,
                "inference_time_sec": per_utt_time,
                "audio_duration_sec": audio_dur,
                "rtf": per_utt_time / audio_dur if audio_dur > 0 else float("inf"),
                "is_cs": is_cs,
                "batch_size": len(kept_indices),
            }
            detail["timings"].append(timing)

            norm_ref = casablanca_normalize(raw_ref)
            # Strip Qwen2-Audio boilerplate BEFORE normalization (see Cell 9b).
            # raw_pred is preserved unchanged in the detail dump for auditability.
            norm_pred = casablanca_normalize(strip_qwen_boilerplate(raw_pred))            
            if not norm_ref:
                continue

            entry = (norm_ref, norm_pred, raw_ref, raw_pred, i)
            detail["full"].append(entry)
            if is_cs:
                detail["code_switched"].append(entry)
            else:
                detail["monolingual"].append(entry)

        if verbose and (batch_end % max(eval_batch_size * 5, 25) == 0 or batch_end == n_samples):
            running_pairs = [(e[0], e[1]) for e in detail["full"]]
            if running_pairs:
                running_wer = jiwer_wer([p[0] for p in running_pairs], [p[1] for p in running_pairs])
                print(f"  [{batch_end}/{n_samples}] running WER={running_wer*100:.2f}%, "
                      f"|CS|={len(detail['code_switched'])}, |Mono|={len(detail['monolingual'])}")

    eval_total_time = time.perf_counter() - t_eval_start

    if was_training:
        model.train()

    # Compute corpus-level metrics for each partition
    metrics: Dict[str, Any] = {}
    for key in ["full", "code_switched", "monolingual"]:
        entries = detail[key]
        if not entries:
            metrics[key] = {"wer": float("nan"), "cer": float("nan"), "n": 0}
            continue
        pairs = [(e[0], e[1]) for e in entries]
        refs = [p[0] for p in pairs]
        hyps = [p[1] for p in pairs]
        m = {
            "wer": float(jiwer_wer(refs, hyps)),
            "cer": float(jiwer_cer(refs, hyps)),
            "n": len(entries),
        }
        if do_bootstrap and len(pairs) >= 10:
            m["wer_ci"] = bootstrap_metric(pairs, jiwer_wer)
            m["cer_ci"] = bootstrap_metric(pairs, jiwer_cer)
        metrics[key] = m

    # Timing summary
    timings = detail["timings"]
    if timings:
        rtfs = [t["rtf"] for t in timings if not np.isinf(t["rtf"])]
        infer_times = [t["inference_time_sec"] for t in timings]
        metrics["timing"] = {
            "n_samples": len(timings),
            "total_eval_time_sec": eval_total_time,
            "mean_inference_time_sec": float(np.mean(infer_times)),
            "median_inference_time_sec": float(np.median(infer_times)),
            "mean_audio_duration_sec": float(np.mean([t["audio_duration_sec"] for t in timings])),
            "mean_rtf": float(np.mean(rtfs)) if rtfs else float("inf"),
            "median_rtf": float(np.median(rtfs)) if rtfs else float("inf"),
        }

    return {"metrics": metrics, "detail": detail}


def print_eval_summary(results: Dict[str, Any], label: str = ""):
    """Pretty-print metrics from evaluate_with_partitioning."""
    m = results["metrics"]
    print(f"\n{'='*70}")
    print(f"  EVALUATION SUMMARY: {label}")
    print(f"{'='*70}")
    for partition in ["full", "code_switched", "monolingual"]:
        p = m.get(partition, {})
        n = p.get("n", 0)
        wer = p.get("wer", float("nan"))
        cer = p.get("cer", float("nan"))
        line = f"  {partition:18s} (n={n:5d})  WER={wer*100:6.2f}%  CER={cer*100:6.2f}%"
        if "wer_ci" in p and p["wer_ci"]:
            ci = p["wer_ci"]
            line += f"  WER 95% CI=[{ci['ci_lower']*100:.2f}%, {ci['ci_upper']*100:.2f}%]"
        print(line)
    if "timing" in m:
        t = m["timing"]
        print(f"  ----")
        print(f"  Mean inference time:     {t['mean_inference_time_sec']:.3f}s/utt")
        print(f"  Mean RTF:                {t['mean_rtf']:.3f}")
        print(f"  Median RTF:              {t['median_rtf']:.3f}")
    n_err = len(results["detail"]["errors"])
    if n_err:
        print(f"  ! Inference errors: {n_err}")
    print(f"{'='*70}")


## 5. Zero-shot baselines

Two baselines reported (per methodology):

1. **Strict prompt** (same as FT) — measures the gap between pretrained-state and the FT regime. This is the WER₀ used for Marco-ASR initial LR.
2. **Darija-specifying prompt** — measures "best zero-shot achievable", since the model knows what to do.

Skip this cell with `MODE in {"wer0_only", "evaluate_only"}`.

In [15]:
# === CELL 14: Zero-shot evaluation on test set ===
zs_strict_results = None
zs_darija_results = None
zs_casa_strict_results = None

if MODE not in ("evaluate_only",):
    print(">>> Zero-shot eval: strict prompt (same as FT prompt) <<<")
    zs_strict_results = evaluate_with_partitioning(
        test_set, model, processor,
        prompt=TASK_PROMPT_FT,
        max_samples=TEST_SET_MAX_SAMPLES,
        eval_batch_size=MARCO_EVAL_BATCH_SIZE,
        do_bootstrap=True,
        verbose=True,
    )
    print_eval_summary(zs_strict_results, label=f"Zero-shot (strict prompt) on MoulSot test ({DATA_SCALE_HOURS}h scale)")

    print("\n>>> Zero-shot eval: Darija-specifying prompt <<<")
    zs_darija_results = evaluate_with_partitioning(
        test_set, model, processor,
        prompt=TASK_PROMPT_ZS_DARIJA,
        max_samples=TEST_SET_MAX_SAMPLES,
        eval_batch_size=MARCO_EVAL_BATCH_SIZE,
        do_bootstrap=True,
        verbose=True,
    )
    print_eval_summary(zs_darija_results, label="Zero-shot (Darija prompt) on MoulSot test")

    # OOD: Casablanca with strict prompt
    print("\n>>> Zero-shot eval: strict prompt on Casablanca (OOD) <<<")
    zs_casa_strict_results = evaluate_with_partitioning(
        casa_eval, model, processor,
        prompt=TASK_PROMPT_FT,
        max_samples=CASABLANCA_OOD_MAX_SAMPLES,
        eval_batch_size=MARCO_EVAL_BATCH_SIZE,
        do_bootstrap=True,
        verbose=True,
    )
    print_eval_summary(zs_casa_strict_results, label="Zero-shot (strict prompt) on Casablanca OOD")

    # Save zero-shot results
    zs_save_path = os.path.join(RESULTS_DIR, f"zero_shot_results_{DATA_SCALE_HOURS}h.json")
    def _strip_detail(r):
        # Drop heavy detail before saving (keeps norm_ref/norm_pred for downstream paired bootstrap)
        if r is None:
            return None
        return {
            "metrics": r["metrics"],
            "detail_compact": {
                k: [{"norm_ref": e[0], "norm_pred": e[1], "raw_ref": e[2], "raw_pred": e[3], "idx": e[4]}
                    for e in r["detail"][k]]
                for k in ["full", "code_switched", "monolingual"]
            },
            "errors": r["detail"]["errors"],
        }
    with open(zs_save_path, "w", encoding="utf-8") as f:
        json.dump({
            "strict_moulsot": _strip_detail(zs_strict_results),
            "darija_moulsot": _strip_detail(zs_darija_results),
            "strict_casablanca": _strip_detail(zs_casa_strict_results),
        }, f, ensure_ascii=False, indent=2)
    print(f"\nZero-shot results saved to {zs_save_path}")
else:
    print(f"MODE={MODE}: skipping zero-shot evals")


>>> Zero-shot eval: strict prompt (same as FT prompt) <<<
Evaluating 1993 samples with prompt='Transcribe speech to text ' (eval_batch_size=4)
Transcription field: 'text'


We detected that you are passing `past_key_values` as a tuple of tuples. This is deprecated and will be removed in v4.47. Please convert your cache or use an appropriate `Cache` class (https://huggingface.co/docs/transformers/kv_cache#legacy-cache-format)


  [100/1993] running WER=156.35%, |CS|=0, |Mono|=100
  [200/1993] running WER=140.10%, |CS|=12, |Mono|=185
  [300/1993] running WER=137.60%, |CS|=24, |Mono|=273
  [400/1993] running WER=136.86%, |CS|=37, |Mono|=359
  [500/1993] running WER=131.03%, |CS|=47, |Mono|=447
  [600/1993] running WER=126.37%, |CS|=59, |Mono|=531
  [700/1993] running WER=123.75%, |CS|=71, |Mono|=618
  [800/1993] running WER=121.80%, |CS|=76, |Mono|=709
  [900/1993] running WER=122.27%, |CS|=85, |Mono|=799
  [1000/1993] running WER=121.13%, |CS|=96, |Mono|=885
  [1100/1993] running WER=120.20%, |CS|=102, |Mono|=979
  [1200/1993] running WER=119.60%, |CS|=109, |Mono|=1070
  [1300/1993] running WER=120.13%, |CS|=116, |Mono|=1161
  [1400/1993] running WER=119.30%, |CS|=123, |Mono|=1253
  [1500/1993] running WER=119.76%, |CS|=135, |Mono|=1341
  [1600/1993] running WER=119.36%, |CS|=145, |Mono|=1430
  [1700/1993] running WER=119.47%, |CS|=157, |Mono|=1517
  [1800/1993] running WER=119.03%, |CS|=166, |Mono|=1605
  [19

## 6. Marco-ASR initialization

### 6.1 Compute WER₀ and initial LR

WER₀ is computed on a dedicated 200-utterance subset of validation (separate from the periodic-eval subset, so the initial LR isn't anchored to the same examples we'll see during training). The initial LR is the input to Marco-ASR's equation (2):

$$\eta_{\text{llm}} = \eta'_{\min} + (\eta'_{\max} - \eta'_{\min}) \cdot \text{clip}(\text{WER}_0, 0, 1)$$

with $\eta'_{\min} = 10^{-6}$ and $\eta'_{\max} = 10^{-4}$ for LLM-based models (per the paper).

In [15]:
# === CELL 15: Compute WER₀ and initial LR ===
def marco_asr_lr(wer_fraction: float, lr_min: float = MARCO_LR_MIN, lr_max: float = MARCO_LR_MAX) -> float:
    """Marco-ASR equation (2) for LLM-based models. wer_fraction in [0, 1]."""
    wer_clipped = max(0.0, min(1.0, wer_fraction))
    return lr_min + (lr_max - lr_min) * wer_clipped


wer0_results = None
initial_lr = MARCO_LR_MAX  # fallback if we skip this cell

if MODE != "evaluate_only":
    print(f">>> Computing WER₀ on {len(wer0_subset)} utterances (strict prompt) <<<")
    wer0_results = evaluate_with_partitioning(
        wer0_subset, model, processor,
        prompt=TASK_PROMPT_FT,
        eval_batch_size=MARCO_EVAL_BATCH_SIZE,
        do_bootstrap=False,  # not needed for a single point
        verbose=False,
    )
    wer0 = wer0_results["metrics"]["full"]["wer"]
    initial_lr = marco_asr_lr(wer0)

    print(f"\nWER₀ (full, strict prompt): {wer0*100:.2f}%")
    print(f"Marco-ASR initial LR:       {initial_lr:.3e}")
    print(f"  formula: {MARCO_LR_MIN:.0e} + ({MARCO_LR_MAX:.0e} - {MARCO_LR_MIN:.0e}) × {wer0:.4f} = {initial_lr:.3e}")

    # Save WER₀ result
    with open(os.path.join(RESULTS_DIR, f"wer0_{DATA_SCALE_HOURS}h.json"), "w", encoding="utf-8") as f:
        json.dump({
            "wer0": wer0,
            "initial_lr": initial_lr,
            "subset_size": len(wer0_subset),
            "prompt": TASK_PROMPT_FT,
            "metrics": wer0_results["metrics"],
        }, f, ensure_ascii=False, indent=2)
else:
    print(f"MODE={MODE}: skipping WER₀ computation")


>>> Computing WER₀ on 200 utterances (strict prompt) <<<
Evaluating 200 samples with prompt='Transcribe speech to text ' (eval_batch_size=4)
Transcription field: 'text'


We detected that you are passing `past_key_values` as a tuple of tuples. This is deprecated and will be removed in v4.47. Please convert your cache or use an appropriate `Cache` class (https://huggingface.co/docs/transformers/kv_cache#legacy-cache-format)



WER₀ (full, strict prompt): 124.79%
Marco-ASR initial LR:       1.000e-04
  formula: 1e-06 + (1e-04 - 1e-06) × 1.2479 = 1.000e-04


## 7. Training preparation

### 7.1 Data collator

Builds the ChatML-formatted conversation with the strict FT prompt, runs it through the processor, and creates labels with prompt masking — loss is computed only on the transcription tokens.

In [16]:
# === CELL 16: Data collator ===
class DarijaASRDataCollator:
    """Data collator for Qwen2-Audio fine-tuning on Darija ASR.

    For each example, builds the conversation:
        user:    [audio_placeholder] [TASK_PROMPT_FT]
        assistant: [transcription]

    Then encodes via processor.apply_chat_template, runs through the processor
    to get input_ids + audio features, and creates labels that mask everything
    before "<|im_start|>assistant\n" with -100 — so the loss is only over
    transcription tokens.
    """

    def __init__(
        self,
        processor,
        task_prompt: str,
        transcription_field: str = "transcription",
        max_audio_length: Optional[int] = None,
        debug: bool = False,
    ):
        self.processor = processor
        self.task_prompt = task_prompt
        self.transcription_field = transcription_field
        self.sampling_rate = processor.feature_extractor.sampling_rate
        self.max_audio_length = max_audio_length or processor.feature_extractor.n_samples
        self.debug = debug

        # Pre-compute the assistant header token sequence
        self._assistant_header = "<|im_start|>assistant\n"
        self._assistant_tokens = processor.tokenizer.encode(
            self._assistant_header, add_special_tokens=False
        )

    def _process_audio(self, audio_dict):
        audio = audio_dict["array"].astype(np.float32, copy=False)
        sr = audio_dict["sampling_rate"]
        if sr != self.sampling_rate:
            audio = librosa.resample(audio, orig_sr=sr, target_sr=self.sampling_rate)
        if len(audio) > self.max_audio_length:
            audio = audio[: self.max_audio_length]
        return audio

    def _resolve_transcription_field(self, example):
        if self.transcription_field in example:
            return self.transcription_field
        for fallback in ["transcription", "text", "transcript", "sentence"]:
            if fallback in example:
                return fallback
        raise KeyError(f"No transcription field found; available keys: {list(example.keys())}")

    def __call__(self, examples):
        audios = []
        combined_texts = []

        for ex in examples:
            try:
                audio = self._process_audio(ex["audio"])
                tfield = self._resolve_transcription_field(ex)
                transcription = ex[tfield]

                conversation = [
                    {
                        "role": "user",
                        "content": [
                            {"type": "audio", "audio_url": "placeholder"},
                            {"type": "text", "text": self.task_prompt},
                        ],
                    },
                    {"role": "assistant", "content": transcription},
                ]
                text = self.processor.apply_chat_template(
                    conversation, add_generation_prompt=False, tokenize=False
                )
                audios.append(audio)
                combined_texts.append(text)
            except Exception as e:
                if self.debug:
                    print(f"[Collator] failed example: {e}")
                continue

        if not combined_texts:
            raise ValueError("No valid examples in batch")

        inputs = self.processor(
            text=list(combined_texts),
            audios=list(audios),
            return_tensors="pt",
            padding=True,
        )

        # Build labels: mask everything up to and including the assistant header
        labels = inputs["input_ids"].clone()
        for i in range(labels.size(0)):
            row = inputs["input_ids"][i].tolist()
            assistant_start = -1
            for j in range(len(row) - len(self._assistant_tokens) + 1):
                if row[j : j + len(self._assistant_tokens)] == self._assistant_tokens:
                    assistant_start = j
                    break
            if assistant_start != -1:
                labels[i, : assistant_start + len(self._assistant_tokens)] = -100
            else:
                # Could not locate assistant header — mask everything (no loss)
                if self.debug:
                    print(f"[Collator] assistant header not found in example {i}; masking all")
                labels[i, :] = -100

        return {
            "input_ids": inputs.input_ids,
            "attention_mask": inputs.attention_mask,
            "input_features": inputs.input_features,
            "feature_attention_mask": inputs.feature_attention_mask,
            "labels": labels,
        }


data_collator = DarijaASRDataCollator(
    processor=processor,
    task_prompt=TASK_PROMPT_FT,
    debug=DEBUG,
)
print("Data collator initialized.")

# Quick sanity check on the collator with one example
if MODE == "fine_tune":
    test_loader = DataLoader(train_set.select(range(1)), batch_size=1, collate_fn=data_collator)
    for batch in test_loader:
        print("\nFirst batch shapes:")
        for k, v in batch.items():
            print(f"  {k}: {tuple(v.shape)} {v.dtype}")
        # Verify the masked labels: only transcription tokens should be != -100
        valid = batch["labels"][0][batch["labels"][0] != -100].tolist()
        if valid:
            target_text = processor.tokenizer.decode(valid, skip_special_tokens=False)
            print(f"\n  Loss target tokens decode to: {target_text[:200]}")
        break
    print("\nCollator sanity check passed ✓")


It is strongly recommended to pass the `sampling_rate` argument to this function. Failing to do so can result in silent errors that might be hard to debug.


Data collator initialized.

First batch shapes:
  input_ids: (1, 44) torch.int64
  attention_mask: (1, 44) torch.int64
  input_features: (1, 128, 3000) torch.float32
  feature_attention_mask: (1, 3000) torch.int32
  labels: (1, 44) torch.int64

  Loss target tokens decode to: انزل تحت، أشوف فيه<|im_end|>


Collator sanity check passed ✓


### 7.2 Apply LoRA

`target_modules="all-linear"` covers every `nn.Linear` in the model except the LM head — that includes the Whisper-LV3 audio encoder, the audio→LLM adapter, and all Qwen2-7B LLM projections. We report the exact trainable parameter count for the methodology audit.

In [17]:
# === CELL 17: Apply LoRA ===
if MODE == "fine_tune":
    lora_config = LoraConfig(
        r=LORA_RANK,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        use_rslora=USE_RSLORA,
        target_modules="all-linear",  # covers encoder + LLM + adapter
        bias=LORA_BIAS,
        task_type="CAUSAL_LM",
    )

    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

    # Save the exact list of LoRA-adapted modules for the methodology audit
    targeted = [n for n, m in model.named_modules() if "lora_A" in n.split(".")[-1] or "lora_B" in n.split(".")[-1]]
    target_audit_path = os.path.join(RESULTS_DIR, f"lora_targets_{DATA_SCALE_HOURS}h.json")
    with open(target_audit_path, "w") as f:
        json.dump({
            "rank": LORA_RANK,
            "alpha": LORA_ALPHA,
            "dropout": LORA_DROPOUT,
            "use_rslora": USE_RSLORA,
            "target_modules_spec": "all-linear",
            "n_lora_module_names": len(targeted),
            "sample_targeted_modules": targeted[:30],
        }, f, indent=2)
    print(f"\n{len(targeted)} LoRA modules injected (encoder + LLM + adapter).")
    print(f"Audit saved to {target_audit_path}")
else:
    print(f"MODE={MODE}: skipping LoRA application")


trainable params: 108,843,008 || all params: 8,505,937,920 || trainable%: 1.2796

836 LoRA modules injected (encoder + LLM + adapter).
Audit saved to /workspace/outputs/results/lora_targets_10h.json


### 7.3 Marco-ASR callback

Implements Algorithm 1 from Ni et al. (2025, §2.1.2) for LLM-based ASR:

1. Every `MARCO_EVAL_STEPS` steps, evaluate WER on the fixed periodic-eval subset.
2. Re-apply equation (2) with the current WER → new LR.
3. Update both `optimizer.param_groups[i].lr` AND `lr_scheduler.base_lrs` so the constant scheduler doesn't overwrite our adjustment on the next step.
4. Track best WER. Stop training when no improvement of ≥ `MARCO_MIN_DELTA_PCT` for `MARCO_PATIENCE` consecutive evals.
5. Save the best LoRA adapter (lowest-WER checkpoint) to `BEST_CKPT_DIR`.

In [18]:
# === CELL 18: Marco-ASR callback ===
class MarcoASRCallback(TrainerCallback):
    """Adaptive LR + early stopping for LLM-based ASR fine-tuning.

    Implements Algorithm 1 from:
        Ni et al., "Marco-ASR: A Principled and Metric-Driven Framework for
        Fine-Tuning Large-Scale ASR Models for Domain Adaptation," 2025.
    """

    def __init__(
        self,
        eval_dataset,
        processor,
        prompt: str,
        eval_steps: int = MARCO_EVAL_STEPS,
        eval_batch_size: int = MARCO_EVAL_BATCH_SIZE,
        lr_min: float = MARCO_LR_MIN,
        lr_max: float = MARCO_LR_MAX,
        patience: int = MARCO_PATIENCE,
        min_delta_pct: float = MARCO_MIN_DELTA_PCT,
        best_checkpoint_dir: Optional[str] = None,
        log_path: Optional[str] = None,
        resume_from_log: bool = False,
        verbose: bool = True,
    ):
        super().__init__()
        self.eval_dataset = eval_dataset
        self.processor = processor
        self.prompt = prompt
        self.eval_steps = eval_steps
        self.eval_batch_size = eval_batch_size
        self.lr_min = lr_min
        self.lr_max = lr_max
        self.patience = patience
        self.min_delta_pct = min_delta_pct
        self.best_checkpoint_dir = best_checkpoint_dir
        self.log_path = log_path
        self.resume_from_log = resume_from_log
        self.verbose = verbose

        self.best_wer = float("inf")
        self.best_step = -1
        self.no_improve_count = 0
        self.history: List[Dict[str, Any]] = []
        self._train_start_time = None

    def _load_resume_state(self):
        """Load history JSON if it exists and reconstruct best_wer + no_improve_count.
        
        Replays the saved history forward, applying the same improvement rule as live
        training, so on_step_end's patience logic continues seamlessly.
        """
        if not self.log_path or not os.path.exists(self.log_path):
            return
        try:
            with open(self.log_path, 'r') as f:
                history = json.load(f)
        except Exception as e:
            if self.verbose:
                print(f"[MARCO-ASR] could not load resume log: {e}")
            return
        if not history:
            return
        best_wer = float('inf')
        best_step = -1
        no_improve = 0
        for record in history:
            wer_pct = record['wer'] * 100
            if wer_pct < (best_wer * 100 - self.min_delta_pct):
                best_wer = record['wer']
                best_step = record['step']
                no_improve = 0
            else:
                no_improve += 1
        self.history = history
        self.best_wer = best_wer
        self.best_step = best_step
        self.no_improve_count = no_improve
        if self.verbose:
            print(f"[MARCO-ASR] resumed from log: {len(history)} prior evals, "
                  f"best WER {best_wer*100:.2f}% at step {best_step}, "
                  f"no_improve_count={no_improve}/{self.patience}")

    def on_train_begin(self, args, state, control, **kwargs):
        self._train_start_time = time.time()
        if self.resume_from_log:
            self._load_resume_state()
        if self.verbose:
            print(f"[MARCO-ASR] training begins; eval every {self.eval_steps} steps "
                  f"on {len(self.eval_dataset)} utterances (batch={self.eval_batch_size})")
        return control

    def _evaluate_wer(self, model) -> Dict[str, float]:
        """Run BATCHED inference on periodic eval subset → corpus-level WER/CER.

        Batching is critical here: at 1000 utterances per eval, sequential inference
        would be ~50min per eval, dominating training time. With batch=8 it's ~12-15min.
        """
        was_training = model.training
        model.eval()

        # Gather all examples first
        all_audios_with_sr = []
        all_refs = []
        for ex in self.eval_dataset:
            try:
                audio = ex["audio"]["array"].astype(np.float32, copy=False)
                sr = ex["audio"]["sampling_rate"]
                ref = ex.get("transcription", ex.get("text", "")) or ""
                all_audios_with_sr.append((audio, sr))
                all_refs.append(ref)
            except Exception as e:
                if self.verbose:
                    print(f"[MARCO-ASR] eval load error: {e}")
                continue

        # Batched inference
        all_preds = []
        for batch_start in range(0, len(all_audios_with_sr), self.eval_batch_size):
            batch = all_audios_with_sr[batch_start : batch_start + self.eval_batch_size]
            try:
                with torch.no_grad():
                    if self.eval_batch_size == 1:
                        audio, sr = batch[0]
                        preds = [run_inference(audio, sr, self.prompt, model=model, processor=self.processor)]
                    else:
                        preds = run_inference_batched(batch, self.prompt, model=model, processor=self.processor)
                all_preds.extend(preds)
            except Exception as e:
                if self.verbose:
                    print(f"[MARCO-ASR] eval inference error at batch {batch_start}: {e}")
                # Pad with empty preds so refs/preds stay aligned
                all_preds.extend([""] * len(batch))

        # Normalize and compute corpus-level metrics.
        # Strip Qwen2-Audio boilerplate (see Cell 9b) so periodic-eval WER reflects
        # transcription quality, not the model's conversational framing.
       
        refs, hyps = [], []
        for ref, pred in zip(all_refs, all_preds):
            ref_n = casablanca_normalize(ref)
            hyp_n = casablanca_normalize(strip_qwen_boilerplate(pred))
            if ref_n:
                refs.append(ref_n)
                hyps.append(hyp_n)

        if was_training:
            model.train()
        if not refs:
            return {"wer": float("nan"), "cer": float("nan"), "n": 0}
        return {
            "wer": float(jiwer_wer(refs, hyps)),
            "cer": float(jiwer_cer(refs, hyps)),
            "n": len(refs),
        }

    @staticmethod
    def _compute_lr(wer_fraction: float, lr_min: float, lr_max: float) -> float:
        wer_clipped = max(0.0, min(1.0, wer_fraction))
        return lr_min + (lr_max - lr_min) * wer_clipped

    def _set_lr(self, optimizer, lr_scheduler, new_lr):
        for pg in optimizer.param_groups:
            pg["lr"] = new_lr
        # Constant scheduler in HF wraps optimizer.param_groups via base_lrs * 1.0,
        # so we must update both to make the change persist past scheduler.step()
        if lr_scheduler is not None and hasattr(lr_scheduler, "base_lrs"):
            lr_scheduler.base_lrs = [new_lr] * len(lr_scheduler.base_lrs)

    def on_step_end(self, args, state, control, model=None, optimizer=None, lr_scheduler=None, **kwargs):
        if state.global_step == 0 or state.global_step % self.eval_steps != 0:
            return control

        # Evaluate
        eval_t0 = time.perf_counter()
        m = self._evaluate_wer(model)
        eval_dt = time.perf_counter() - eval_t0
        wer = m["wer"]
        cer = m["cer"]

        # Adjust LR via Marco-ASR equation (2) with current WER
        new_lr = self._compute_lr(wer, self.lr_min, self.lr_max)
        self._set_lr(optimizer, lr_scheduler, new_lr)

        # History
        record = {
            "step": int(state.global_step),
            "wer": wer,
            "cer": cer,
            "lr": new_lr,
            "eval_time_sec": eval_dt,
            "elapsed_sec": time.time() - (self._train_start_time or time.time()),
            "timestamp": time.time(),
        }
        self.history.append(record)
        if self.log_path:
            try:
                with open(self.log_path, "w", encoding="utf-8") as f:
                    json.dump(self.history, f, indent=2)
            except Exception:
                pass

        # Early stopping logic + save best checkpoint
        wer_pct = wer * 100
        improved = wer_pct < (self.best_wer * 100 - self.min_delta_pct)
        if improved:
            self.best_wer = wer
            self.best_step = int(state.global_step)
            self.no_improve_count = 0
            if self.best_checkpoint_dir:
                try:
                    model.save_pretrained(self.best_checkpoint_dir)
                except Exception as e:
                    if self.verbose:
                        print(f"[MARCO-ASR] could not save checkpoint: {e}")
        else:
            self.no_improve_count += 1
            if self.no_improve_count >= self.patience:
                if self.verbose:
                    print(f"[MARCO-ASR] early stopping at step {state.global_step}: "
                          f"no improvement (>={self.min_delta_pct} pts) for "
                          f"{self.patience} consecutive evaluations")
                control.should_training_stop = True

        if self.verbose:
            arrow = "↓" if improved else "→"
            print(f"[MARCO-ASR step={state.global_step:5d}] {arrow} "
                  f"WER={wer_pct:6.2f}%  CER={cer*100:6.2f}%  "
                  f"LR={new_lr:.2e}  best={self.best_wer*100:6.2f}% (step {self.best_step})  "
                  f"no_improve={self.no_improve_count}/{self.patience}  "
                  f"eval={eval_dt:.1f}s")

        return control

    def on_train_end(self, args, state, control, **kwargs):
        if self.verbose:
            print(f"\n[MARCO-ASR] training ended. Best WER: {self.best_wer*100:.2f}% at step {self.best_step}")
        return control


print("MarcoASRCallback class ready.")


MarcoASRCallback class ready.


### 7.4 Training arguments

`lr_scheduler_type="constant"` is essential — it ensures Marco-ASR's adaptive LR adjustments are not overwritten by a built-in scheduler. The initial LR is the value computed from WER₀ in Cell 15.

`eval_strategy="no"` because we run our own evaluation inside the Marco-ASR callback (running `model.generate` for ASR is not what HF's default eval expects, and bypassing it is cleaner than fighting `predict_with_generate`).

In [19]:
# === CELL 19: Training arguments ===
if MODE == "fine_tune":
    run_name = f"qwen2audio-darija-marco-{DATA_SCALE_HOURS}h-r{LORA_RANK}-lr{initial_lr:.1e}"

    training_args = TrainingArguments(
        output_dir=TRAINER_OUTPUT_DIR,
        run_name=run_name,
        # --- Data / batch ---
        per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        num_train_epochs=NUM_TRAIN_EPOCHS,
        # --- Optimizer ---
        learning_rate=initial_lr,                  # from Marco-ASR formula
        weight_decay=WEIGHT_DECAY,
        lr_scheduler_type="constant",              # Marco-ASR overrides via callback
        # --- Precision / memory ---
        bf16=True,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        # --- Eval / saving ---
        eval_strategy="no",                        # Marco-ASR runs its own eval
        save_strategy="steps",
        #save_steps=max(MARCO_EVAL_STEPS * 5, 500),
        #save_total_limit=2,
        save_steps=max(MARCO_EVAL_STEPS * 2, 200),    # save every 200 steps (was 500)
        save_total_limit=3,                             # keep 3 recent checkpoints (was 2)
        save_safetensors=True,                          # smaller atomic writes
        # --- Logging ---
        logging_steps=10,
        logging_dir=os.path.join(LOGS_DIR, run_name),
        report_to="tensorboard",
        # --- Misc ---
        remove_unused_columns=False,
        dataloader_num_workers=0,
        seed=SEED,
        # Hub
        hub_model_id=HUB_MODEL_ID,
        push_to_hub=(HUB_MODEL_ID is not None),
    )

    # Marco-ASR callback
    marco_callback = MarcoASRCallback(
        eval_dataset=periodic_eval_subset,
        processor=processor,
        prompt=TASK_PROMPT_FT,
        eval_steps=MARCO_EVAL_STEPS,
        eval_batch_size=MARCO_EVAL_BATCH_SIZE,
        lr_min=MARCO_LR_MIN,
        lr_max=MARCO_LR_MAX,
        patience=MARCO_PATIENCE,
        min_delta_pct=MARCO_MIN_DELTA_PCT,
        best_checkpoint_dir=BEST_CKPT_DIR,
        log_path=os.path.join(LOGS_DIR, f"marco_history_{DATA_SCALE_HOURS}h.json"),
        resume_from_log=RESUME_FROM_CHECKPOINT,
        verbose=True,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        data_collator=data_collator,
        train_dataset=train_set,
        callbacks=[marco_callback],
    )

    print(f"Trainer initialized.")
    print(f"  Run name:                 {run_name}")
    print(f"  Initial LR (from WER₀):   {initial_lr:.3e}")
    print(f"  Train samples:            {len(train_set)} (~{actual_train_hours:.2f}h)")
    print(f"  Per-device batch:         {PER_DEVICE_BATCH_SIZE}")
    print(f"  Grad accum:               {GRADIENT_ACCUMULATION_STEPS}")
    print(f"  Effective batch:          {PER_DEVICE_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")
    print(f"  Epochs (max):             {NUM_TRAIN_EPOCHS} (Marco-ASR may stop earlier)")
    print(f"  Eval / LR adjust every:   {MARCO_EVAL_STEPS} steps on {MARCO_EVAL_SUBSET_SIZE} utt")
    print(f"  Eval batch size:          {MARCO_EVAL_BATCH_SIZE}")
else:
    print(f"MODE={MODE}: skipping trainer setup")


Trainer initialized.
  Run name:                 qwen2audio-darija-marco-10h-r32-lr1.0e-04
  Initial LR (from WER₀):   1.000e-04
  Train samples:            7899 (~10.00h)
  Per-device batch:         4
  Grad accum:               4
  Effective batch:          16
  Epochs (max):             3 (Marco-ASR may stop earlier)
  Eval / LR adjust every:   100 steps on 200 utt
  Eval batch size:          4


In [20]:
# === CELL 19b: Disk-space guard callback ===
import shutil
from transformers import TrainerCallback

class DiskSpaceGuardCallback(TrainerCallback):
    """Aborts training cleanly BEFORE a save would fail due to disk-fill.

    The actual problem we had: torch.save() crashed mid-write, leaving a
    corrupt partial checkpoint AND no clean exit. This callback fires
    before each save and checks if there's enough headroom. If not, it
    aborts training cleanly so the previous (intact) checkpoint stays
    usable for resume.
    """
    def __init__(self, output_dir, min_free_gb=8.0, verbose=True):
        self.output_dir = output_dir
        self.min_free_gb = min_free_gb
        self.verbose = verbose

    def _check_disk(self, label="save"):
        free_gb = shutil.disk_usage(self.output_dir).free / 1e9
        if self.verbose:
            print(f"[DISK-GUARD] before {label}: {free_gb:.1f} GB free in {self.output_dir}")
        return free_gb

    def on_save(self, args, state, control, **kwargs):
        free_gb = self._check_disk(f"save at step {state.global_step}")
        if free_gb < self.min_free_gb:
            print(f"[DISK-GUARD] ABORTING: only {free_gb:.1f} GB free, need ≥ {self.min_free_gb} GB")
            print(f"[DISK-GUARD] The previous checkpoint (1 save_steps earlier) is intact and resumable.")
            print(f"[DISK-GUARD] Free space (HF cache, old runs) or expand the volume, then re-run Cell 20.")
            control.should_training_stop = True
        return control

    def on_train_begin(self, args, state, control, **kwargs):
        self._check_disk("training start")
        return control

# Instantiate and ADD to the trainer's callbacks
disk_guard = DiskSpaceGuardCallback(OUTPUT_DIR, min_free_gb=8.0)
trainer.add_callback(disk_guard)
print("DiskSpaceGuardCallback attached to trainer.")

DiskSpaceGuardCallback attached to trainer.


### 7.5 Train

In [27]:
# === CELL 20: Run training (with pre-flight check + auto-fallback resume) ===
import shutil
import os

if MODE == "fine_tune":
    # === Pre-flight checks ===
    free_gb = shutil.disk_usage(OUTPUT_DIR).free / 1e9
    print(f"[PRE-FLIGHT] Free disk in {OUTPUT_DIR}: {free_gb:.1f} GB")
    if free_gb < 15:
        raise RuntimeError(
            f"Pre-flight failed: only {free_gb:.1f} GB free; need ≥ 15 GB for training. "
            f"Expand the network volume in RunPod's dashboard or free up space."
        )

    def get_latest_checkpoint(output_dir):
        if not os.path.isdir(output_dir):
            return None
        candidates = [
            d for d in os.listdir(output_dir)
            if d.startswith("checkpoint-") and os.path.isdir(os.path.join(output_dir, d))
        ]
        if not candidates:
            return None
        candidates.sort(key=lambda x: int(x.split("-")[-1]))
        return os.path.join(output_dir, candidates[-1])

    def is_valid_checkpoint(ckpt_path):
        """A trainer checkpoint is loadable iff it has all the small metadata files
        (which are written LAST in the save sequence — if they're present, the save
        completed; if missing, the save was interrupted)."""
        required = ["trainer_state.json", "adapter_config.json"]
        return all(os.path.exists(os.path.join(ckpt_path, f)) for f in required)

    def train_with_auto_fallback(trainer, output_dir, max_fallbacks=3):
        """Train with automatic resume from latest checkpoint, falling back to
        earlier checkpoints if the latest is corrupted."""
        import time
        for attempt in range(max_fallbacks + 1):
            resume_path = get_latest_checkpoint(output_dir)
            if resume_path is None:
                print(f"[FALLBACK] No checkpoints found. Starting from scratch.")
                return trainer.train()

            if not is_valid_checkpoint(resume_path):
                print(f"[FALLBACK] {resume_path} is incomplete (missing trainer_state.json or adapter_config.json).")
                corrupted_marker = resume_path + f".corrupted_{int(time.time())}"
                os.rename(resume_path, corrupted_marker)
                print(f"[FALLBACK] Moved to {corrupted_marker}; retrying with previous checkpoint...")
                continue

            print(f"[FALLBACK] Resuming from {resume_path} (attempt {attempt+1}/{max_fallbacks+1})...")
            try:
                return trainer.train(resume_from_checkpoint=resume_path)
            except (RuntimeError, FileNotFoundError, EOFError) as e:
                print(f"[FALLBACK] Resume from {resume_path} failed: {type(e).__name__}: {e}")
                if attempt >= max_fallbacks:
                    print(f"[FALLBACK] Exhausted retry attempts. Raising.")
                    raise
                corrupted_marker = resume_path + f".corrupted_{int(time.time())}"
                os.rename(resume_path, corrupted_marker)
                print(f"[FALLBACK] Moved corrupted checkpoint to {corrupted_marker}")
                print(f"[FALLBACK] Retrying with next-latest checkpoint...")
        # Unreachable but explicit
        raise RuntimeError("train_with_auto_fallback exited without completing")

    print(f"\n>>> Starting fine-tuning ({DATA_SCALE_HOURS}h, r={LORA_RANK}) <<<\n")
    train_t0 = time.time()
    train_result = train_with_auto_fallback(trainer, TRAINER_OUTPUT_DIR, max_fallbacks=3)
    train_total = time.time() - train_t0

    # === Final disk check + saves (unchanged from before) ===
    free_after = shutil.disk_usage(OUTPUT_DIR).free / 1e9
    print(f"\n[POST-TRAIN] Free disk: {free_after:.1f} GB")

    final_dir = os.path.join(OUTPUT_DIR, f"final_{DATA_SCALE_HOURS}h")
    trainer.save_model(final_dir)
    processor.save_pretrained(final_dir)
    print(f"\nFinal model saved to: {final_dir}")
    print(f"Best WER checkpoint:  {BEST_CKPT_DIR} (WER {marco_callback.best_wer*100:.2f}% at step {marco_callback.best_step})")

    train_cost = {
        "data_scale_hours": DATA_SCALE_HOURS,
        "actual_train_hours": actual_train_hours,
        "n_train_samples": len(train_set),
        "wall_clock_sec": train_total,
        "wall_clock_hours": train_total / 3600,
        "best_wer_periodic_eval": marco_callback.best_wer,
        "best_step": marco_callback.best_step,
        "global_step_at_end": trainer.state.global_step,
        "initial_lr": initial_lr,
        "wer0": wer0_results["metrics"]["full"]["wer"] if wer0_results else None,
    }
    with open(os.path.join(RESULTS_DIR, f"training_cost_{DATA_SCALE_HOURS}h.json"), "w") as f:
        json.dump(train_cost, f, indent=2)
    print(f"\nTraining cost: {train_total/3600:.2f} GPU-hours")
else:
    print(f"MODE={MODE}: skipping training")

[PRE-FLIGHT] Free disk in /workspace/outputs: 194289.7 GB

>>> Starting fine-tuning (10h, r=32) <<<

[FALLBACK] Resuming from /workspace/outputs/trainer_10h/checkpoint-200 (attempt 1/4)...


It is strongly recommended to pass the `sampling_rate` argument to this function. Failing to do so can result in silent errors that might be hard to debug.
It is strongly recommended to pass the `sampling_rate` argument to this function. Failing to do so can result in silent errors that might be hard to debug.
It is strongly recommended to pass the `sampling_rate` argument to this function. Failing to do so can result in silent errors that might be hard to debug.


[MARCO-ASR] training begins; eval every 100 steps on 200 utterances (batch=4)
[DISK-GUARD] before training start: 194289.5 GB free in /workspace/outputs


It is strongly recommended to pass the `sampling_rate` argument to this function. Failing to do so can result in silent errors that might be hard to debug.
It is strongly recommended to pass the `sampling_rate` argument to this function. Failing to do so can result in silent errors that might be hard to debug.


Step,Training Loss
210,1.252100
220,1.225400
230,1.217000
240,1.136700
250,1.238100
260,1.234300
270,1.144500
280,1.066000
290,1.125300
300,1.152600


...

[DISK-GUARD] before save at step 800: 194526.4 GB free in /workspace/outputs


It is strongly recommended to pass the `sampling_rate` argument to this function. Failing to do so can result in silent errors that might be hard to debug.
It is strongly recommended to pass the `sampling_rate` argument to this function. Failing to do so can result in silent errors that might be hard to debug.
It is strongly recommended to pass the `sampling_rate` argument to this function. Failing to do so can result in silent errors that might be hard to debug.
It is strongly recommended to pass the `sampling_rate` argument to this function. Failing to do so can result in silent errors that might be hard to debug.
It is strongly recommended to pass the `sampling_rate` argument to this function. Failing to do so can result in silent errors that might be hard to debug.
It is strongly recommended to pass the `sampling_rate` argument to this function. Failing to do so can result in silent errors that might be hard to debug.
It is strongly recommended to pass the `sampling_rate` argument 

[MARCO-ASR] early stopping at step 900: no improvement (>=0.5 pts) for 3 consecutive evaluations
[MARCO-ASR step=  900] → WER= 50.47%  CER= 21.51%  LR=5.10e-05  best= 50.24% (step 600)  no_improve=3/3  eval=188.7s

[MARCO-ASR] training ended. Best WER: 50.24% at step 600

[POST-TRAIN] Free disk: 194197.9 GB


/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:227: UserWarning: Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.
  warnings.warn("Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.")



Final model saved to: /workspace/outputs/final_10h
Best WER checkpoint:  /workspace/outputs/best_marco_asr_10h (WER 50.24% at step 600)

Training cost: 2.09 GPU-hours


In [26]:
import torch
import functools

if not getattr(torch.load, "_patched_for_hf_trainer", False):
    _original_torch_load = torch.load

    @functools.wraps(_original_torch_load)
    def _patched_torch_load(f, *args, **kwargs):
        # Default to weights_only=False to restore PyTorch <2.6 behavior for
        # HF Trainer's rng_state.pth (numpy state objects fail the new strict default).
        # Callers can still opt into weights_only=True by passing it explicitly.
        kwargs.setdefault("weights_only", False)
        return _original_torch_load(f, *args, **kwargs)

    _patched_torch_load._patched_for_hf_trainer = True
    torch.load = _patched_torch_load
    print("✓ Patched torch.load to default weights_only=False (HF Trainer rng_state compat)")
else:
    print("Already patched, no-op")

✓ Patched torch.load to default weights_only=False (HF Trainer rng_state compat)


In [23]:
import os
import shutil

# 2a. Verify the trainer checkpoint we'll resume from is complete
ckpt_200 = os.path.join(TRAINER_OUTPUT_DIR, "checkpoint-200")
if os.path.exists(ckpt_200):
    required = ["trainer_state.json", "adapter_config.json", "optimizer.pt"]
    missing = [f for f in required if not os.path.exists(os.path.join(ckpt_200, f))]
    if missing:
        print(f"⚠ checkpoint-200 incomplete (missing {missing}). Resume will fall back further or start from scratch.")
    else:
        print(f"✓ checkpoint-200 looks complete — resume will pick up from step 200")
else:
    print(f"✗ No checkpoint-200 found (save_steps may have not fired yet). Training will start from scratch.")
    print("  Cost: ~10 min of compute lost (200 steps × ~3s/step).")

# 2b. Move the stale history JSON aside (don't delete — keep as audit trail)
history_path = os.path.join(LOGS_DIR, f"marco_history_{DATA_SCALE_HOURS}h.json")
if os.path.exists(history_path):
    backup_path = history_path + ".stale_from_previous_run"
    shutil.move(history_path, backup_path)
    print(f"✓ Stale history → {backup_path}")
else:
    print(f"  (no history file at {history_path} — nothing to move)")

# 2c. Reset the in-memory callback state so resume starts clean
marco_callback.best_wer = float("inf")
marco_callback.best_step = -1
marco_callback.no_improve_count = 0
marco_callback.history = []
marco_callback._train_start_time = None
print("✓ Reset marco_callback in-memory state: best_wer=inf, no_improve=0, history=[]")

✓ checkpoint-200 looks complete — resume will pick up from step 200
✓ Stale history → /workspace/outputs/logs/marco_history_10h.json.stale_from_previous_run
✓ Reset marco_callback in-memory state: best_wer=inf, no_improve=0, history=[]


In [23]:
import torch, os
from safetensors.torch import load_file as load_safetensors

CKPT = "/workspace/outputs/trainer_10h/checkpoint-500"

# Test 1: adapter weights
print("Testing adapter_model.safetensors...")
try:
    adapter_state = load_safetensors(os.path.join(CKPT, "adapter_model.safetensors"))
    total = sum(v.numel() for v in adapter_state.values())
    print(f"  ✓ adapter OK: {len(adapter_state)} tensors, {total/1e6:.1f}M params total")
except Exception as e:
    print(f"  ✗ adapter CORRUPTED: {type(e).__name__}: {e}")

# Test 2: optimizer state — this is the critical one
print("\nTesting optimizer.pt (this is the file that was being written when disk filled)...")
try:
    opt_state = torch.load(os.path.join(CKPT, "optimizer.pt"), weights_only=False)
    print(f"  ✓ optimizer OK: param_groups={len(opt_state.get('param_groups', []))}, "
          f"state_entries={len(opt_state.get('state', {}))}")
except Exception as e:
    print(f"  ✗ optimizer CORRUPTED: {type(e).__name__}: {e}")
    print("  → fallback path required (see below)")

Testing adapter_model.safetensors...
  ✓ adapter OK: 837 tensors, 748.0M params total

Testing optimizer.pt (this is the file that was being written when disk filled)...
  ✗ optimizer CORRUPTED: RuntimeError: PytorchStreamReader failed reading zip archive: failed finding central directory
  → fallback path required (see below)


## 8. Post-training evaluation

Loads the best-by-WER checkpoint (saved by the Marco-ASR callback) and runs the full evaluation protocol on:

1. MoulSot test set (in-distribution)
2. Casablanca Morocco test slice (OOD)

With code-switching partitioning and 95% bootstrap CIs throughout.

In [28]:
# === CELL 21: Load best checkpoint and run final evaluation ===
if MODE in ("fine_tune", "evaluate_only"):
    # Determine which adapter to load
    if MODE == "evaluate_only":
        adapter_path = EVAL_ADAPTER_PATH
        print(f"MODE=evaluate_only: loading adapter from {adapter_path}")
        # In evaluate_only mode, the model is the base — apply the adapter now.
        # If the model is already PEFT-wrapped from a previous run, this needs care;
        # reset by reloading if necessary.
        from peft import PeftModel
        if not isinstance(model, PeftModel):
            model = PeftModel.from_pretrained(model, adapter_path)
        else:
            # Already wrapped — load adapter weights into the existing wrapper
            model.load_adapter(adapter_path, "default")
            model.set_adapter("default")
        model.to(DEVICE).eval()
    else:
        adapter_path = BEST_CKPT_DIR
        if os.path.exists(adapter_path):
            print(f"Loading best Marco-ASR checkpoint from {adapter_path} (WER {marco_callback.best_wer*100:.2f}%)")
            model.load_adapter(adapter_path, "best_marco")
            model.set_adapter("best_marco")
            model.eval()
        else:
            print(f"WARNING: no best checkpoint at {adapter_path}; using final model")
            model.eval()

    # Final eval — MoulSot test
    print("\n>>> Final eval: MoulSot test (in-distribution) <<<")
    ft_test_results = evaluate_with_partitioning(
        test_set, model, processor,
        prompt=TASK_PROMPT_FT,
        max_samples=TEST_SET_MAX_SAMPLES,
        eval_batch_size=MARCO_EVAL_BATCH_SIZE,
        do_bootstrap=True,
        verbose=True,
    )
    print_eval_summary(ft_test_results, label=f"Fine-tuned ({DATA_SCALE_HOURS}h) on MoulSot test")

    # Final eval — Casablanca OOD
    print("\n>>> Final eval: Casablanca Morocco (OOD) <<<")
    ft_casa_results = evaluate_with_partitioning(
        casa_eval, model, processor,
        prompt=TASK_PROMPT_FT,
        max_samples=CASABLANCA_OOD_MAX_SAMPLES,
        eval_batch_size=MARCO_EVAL_BATCH_SIZE,
        do_bootstrap=True,
        verbose=True,
    )
    print_eval_summary(ft_casa_results, label=f"Fine-tuned ({DATA_SCALE_HOURS}h) on Casablanca OOD")

    # Save final results
    def _strip_detail(r):
        if r is None:
            return None
        return {
            "metrics": r["metrics"],
            "detail_compact": {
                k: [{"norm_ref": e[0], "norm_pred": e[1], "raw_ref": e[2], "raw_pred": e[3], "idx": e[4]}
                    for e in r["detail"][k]]
                for k in ["full", "code_switched", "monolingual"]
            },
            "errors": r["detail"]["errors"],
        }

    final_path = os.path.join(RESULTS_DIR, f"final_results_{DATA_SCALE_HOURS}h.json")
    with open(final_path, "w", encoding="utf-8") as f:
        json.dump({
            "moulsot_test": _strip_detail(ft_test_results),
            "casablanca_ood": _strip_detail(ft_casa_results),
        }, f, ensure_ascii=False, indent=2)
    print(f"\nFinal results saved to {final_path}")
else:
    print(f"MODE={MODE}: skipping final evaluation")


Loading best Marco-ASR checkpoint from /workspace/outputs/best_marco_asr_10h (WER 50.24%)

>>> Final eval: MoulSot test (in-distribution) <<<
Evaluating 1993 samples with prompt='Transcribe speech to text ' (eval_batch_size=4)
Transcription field: 'text'
  [100/1993] running WER=54.20%, |CS|=0, |Mono|=100
  [200/1993] running WER=56.25%, |CS|=12, |Mono|=185
  [300/1993] running WER=56.22%, |CS|=24, |Mono|=273
  [400/1993] running WER=51.43%, |CS|=37, |Mono|=359
  [500/1993] running WER=54.76%, |CS|=47, |Mono|=447
  [600/1993] running WER=55.82%, |CS|=59, |Mono|=531
  [700/1993] running WER=56.33%, |CS|=71, |Mono|=618
  [800/1993] running WER=55.93%, |CS|=76, |Mono|=709
  [900/1993] running WER=55.95%, |CS|=85, |Mono|=799
  [1000/1993] running WER=56.10%, |CS|=96, |Mono|=885
  [1100/1993] running WER=56.35%, |CS|=102, |Mono|=979
  [1200/1993] running WER=56.59%, |CS|=109, |Mono|=1070
  [1300/1993] running WER=56.78%, |CS|=116, |Mono|=1161
  [1400/1993] running WER=57.14%, |CS|=123, |Mon

## 9. Paired bootstrap: zero-shot vs fine-tuned

Computes the WER difference between zero-shot (strict prompt) and fine-tuned, with paired bootstrap to assess statistical significance. A 95% CI on the difference that excludes zero is the methodologically defensible signal that fine-tuning improved transcription.

In [29]:
# === CELL 22: Paired bootstrap (zero-shot vs fine-tuned) ===
if MODE == "fine_tune" and zs_strict_results is not None and ft_test_results is not None:
    # Align by sample index — both eval'd the same test_set in order
    def to_paired_lookup(res):
        return {e[4]: (e[0], e[1]) for e in res["detail"]["full"]}

    zs_lookup = to_paired_lookup(zs_strict_results)
    ft_lookup = to_paired_lookup(ft_test_results)

    common_idx = sorted(set(zs_lookup.keys()) & set(ft_lookup.keys()))
    if not common_idx:
        print("WARNING: no overlapping indices; cannot do paired bootstrap")
    else:
        zs_pairs = [zs_lookup[i] for i in common_idx]
        ft_pairs = [ft_lookup[i] for i in common_idx]
        wer_diff = paired_bootstrap_diff(zs_pairs, ft_pairs, jiwer_wer)
        cer_diff = paired_bootstrap_diff(zs_pairs, ft_pairs, jiwer_cer)

        zs_w = jiwer_wer([p[0] for p in zs_pairs], [p[1] for p in zs_pairs])
        ft_w = jiwer_wer([p[0] for p in ft_pairs], [p[1] for p in ft_pairs])

        print(f"\nPaired comparison on {len(common_idx)} test utterances:")
        print(f"  Zero-shot WER (strict): {zs_w*100:.2f}%")
        print(f"  Fine-tuned WER:         {ft_w*100:.2f}%")
        print(f"  Δ WER (FT − ZS):        {wer_diff['mean_diff']*100:+.2f} pts  "
              f"95% CI: [{wer_diff['ci_lower']*100:+.2f}, {wer_diff['ci_upper']*100:+.2f}]  "
              f"{'[significant]' if wer_diff['significant'] else '[NOT significant]'}")
        print(f"  Δ CER (FT − ZS):        {cer_diff['mean_diff']*100:+.2f} pts  "
              f"95% CI: [{cer_diff['ci_lower']*100:+.2f}, {cer_diff['ci_upper']*100:+.2f}]  "
              f"{'[significant]' if cer_diff['significant'] else '[NOT significant]'}")

        with open(os.path.join(RESULTS_DIR, f"paired_bootstrap_{DATA_SCALE_HOURS}h.json"), "w") as f:
            json.dump({
                "n_paired": len(common_idx),
                "zs_wer": zs_w,
                "ft_wer": ft_w,
                "wer_diff": wer_diff,
                "cer_diff": cer_diff,
            }, f, indent=2)
else:
    print(f"MODE={MODE}: skipping paired bootstrap")


NameError: name 'zs_strict_results' is not defined

In [ ]:
# === CELL 23 (new): Marco-ASR training curves ===
import matplotlib.pyplot as plt
import json
import os

def plot_marco_history(log_path, save_path=None, show=True):
    """Plot WER, LR, and patience over training steps.

    Safe to call mid-training — reads the on-disk JSON which is rewritten
    after every periodic eval. Will work even from a separate kernel
    (e.g., inspect progress while training runs).
    """
    if not os.path.exists(log_path):
        print(f"No history found at {log_path}")
        return
    with open(log_path, "r") as f:
        history = json.load(f)
    if not history:
        print("History is empty (no evals yet)")
        return

    steps = [h["step"] for h in history]
    wers = [h["wer"] * 100 for h in history]
    cers = [h["cer"] * 100 for h in history]
    lrs = [h["lr"] for h in history]
    elapsed = [h["elapsed_sec"] / 60 for h in history]  # in minutes

    # Best WER trajectory (running min)
    best_wer = float("inf")
    best_wer_curve = []
    best_steps = []
    for s, w in zip(steps, wers):
        if w < best_wer:
            best_wer = w
            best_steps.append((s, w))
        best_wer_curve.append(best_wer)

    fig, axes = plt.subplots(2, 2, figsize=(14, 8))
    fig.suptitle(f"Marco-ASR training curves ({DATA_SCALE_HOURS}h scale, {len(history)} evals)",
                 fontsize=13, fontweight="bold")

    # --- 1. WER + best-so-far ---
    ax = axes[0, 0]
    ax.plot(steps, wers, "o-", color="#1f77b4", alpha=0.7, label="WER per eval", markersize=4)
    ax.plot(steps, best_wer_curve, "-", color="#d62728", linewidth=2, label="best WER so far")
    if best_steps:
        s_best, w_best = best_steps[-1]
        ax.scatter([s_best], [w_best], color="#d62728", s=80, zorder=5,
                   label=f"current best: {w_best:.2f}% @ step {s_best}")
    ax.set_xlabel("training step")
    ax.set_ylabel("WER (%)")
    ax.set_title("WER vs. step")
    ax.legend(loc="upper right", fontsize=9)
    ax.grid(alpha=0.3)

    # --- 2. CER ---
    ax = axes[0, 1]
    ax.plot(steps, cers, "o-", color="#2ca02c", alpha=0.7, markersize=4)
    ax.set_xlabel("training step")
    ax.set_ylabel("CER (%)")
    ax.set_title("CER vs. step")
    ax.grid(alpha=0.3)

    # --- 3. LR (Marco-ASR adapts based on WER) ---
    ax = axes[1, 0]
    ax.plot(steps, lrs, "o-", color="#ff7f0e", alpha=0.8, markersize=4)
    ax.set_xlabel("training step")
    ax.set_ylabel("learning rate")
    ax.set_yscale("log")
    ax.set_title("LR vs. step (Marco-ASR adaptive)")
    ax.grid(alpha=0.3, which="both")

    # --- 4. Wall-clock + eval cost over time ---
    ax = axes[1, 1]
    ax.plot(elapsed, wers, "o-", color="#9467bd", alpha=0.7, markersize=4, label="WER")
    ax.set_xlabel("wall-clock elapsed (minutes)")
    ax.set_ylabel("WER (%)")
    ax.set_title("WER vs. wall-clock time")
    ax.grid(alpha=0.3)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=120, bbox_inches="tight")
        print(f"Saved to {save_path}")
    if show:
        plt.show()
    plt.close()

    # Text summary
    print(f"\n  Total evals so far:    {len(history)}")
    print(f"  Best WER:              {min(wers):.2f}% at step {steps[wers.index(min(wers))]}")
    print(f"  Latest WER:            {wers[-1]:.2f}% at step {steps[-1]}")
    print(f"  Latest LR:             {lrs[-1]:.2e}")
    print(f"  Latest elapsed:        {elapsed[-1]:.1f} min")

# Auto-call on the current run
log_path = os.path.join(LOGS_DIR, f"marco_history_{DATA_SCALE_HOURS}h.json")
plot_path = os.path.join(RESULTS_DIR, f"marco_curves_{DATA_SCALE_HOURS}h.png")
plot_marco_history(log_path, save_path=plot_path, show=True)

## 10. Summary

What this notebook produces (under `OUTPUT_DIR`):

- `results/zero_shot_results_{H}h.json` — zero-shot WER/CER for both prompts on MoulSot test + Casablanca OOD, with bootstrap CIs and per-utterance pairs (for downstream paired-bootstrap analysis).
- `results/wer0_{H}h.json` — WER₀ value and computed initial LR.
- `results/lora_targets_{H}h.json` — exact LoRA module audit (count + sample names).
- `results/training_cost_{H}h.json` — wall-clock GPU-hours, total step count, best step.
- `results/final_results_{H}h.json` — post-FT WER/CER on MoulSot test + Casablanca OOD, with bootstrap CIs.
- `results/paired_bootstrap_{H}h.json` — paired Δ WER between zero-shot and fine-tuned, with significance.
- `logs/marco_history_{H}h.json` — full Marco-ASR trajectory: step, WER, CER, LR, eval time, wall-clock elapsed.
- `best_marco_asr_{H}h/` — LoRA adapter at the lowest periodic-eval WER step.
- `final_{H}h/` — LoRA adapter at the end of training.

The methodology audit answers (per Phase 1 §8):

- Trainable parameter count → `lora_targets_{H}h.json`
- Compute consumed → `training_cost_{H}h.json`
- WER₀ as initial LR seed → `wer0_{H}h.json`
- Code-switching partitioning → `final_results_{H}h.json` (key: `code_switched` and `monolingual`)
- OOD generalization → `final_results_{H}h.json` (key: `casablanca_ood`)
- Statistical significance → `paired_bootstrap_{H}h.json`

To run at all three data scales (10h, 30h, 80h), set `DATA_SCALE_HOURS` at the top and re-execute.